In [0]:
# %run ../ntb_ia_utils

In [0]:
%pip install --upgrade pip

In [0]:
%pip install transformers==4.16.0 # << versao requirements

In [0]:
%pip install tensorflow==2.11.0 # << versao requirements

In [0]:
%pip install striprtf==0.0.20 # << versao requirements

In [0]:
%pip install spacy==3.5.3 spacy-udpipe==1.0.0 conllu==4.4.1 nltk==3.7

In [0]:
%pip install pytorch-lightning==1.4.9 torchmetrics==0.5.1 # << versao original

In [0]:
%pip install "protobuf==3.20.3" # << restaurar apos a tentativa 01

In [0]:
%pip install https://github.com/explosion/spacy-models/releases/download/pt_core_news_sm-3.5.0/pt_core_news_sm-3.5.0-py3-none-any.whl # << usado na tentativa 01 versao requirements

In [0]:
%pip install https://github.com/explosion/spacy-models/releases/download/pt_core_news_lg-3.5.0/pt_core_news_lg-3.5.0-py3-none-any.whl # << usado na tentativa 01 versao requirements

In [0]:
%pip uninstall -y sentence-transformers transformers huggingface_hub
%pip install huggingface_hub==0.16.4
%pip install transformers==4.32.0
%pip install sentence-transformers==2.2.2

In [0]:
dbutils.library.restartPython()

In [0]:
from sentence_transformers import SentenceTransformer # <<
model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
print("✅ Modelo carregado com sucesso!")

In [0]:
!pip install Unidecode

In [0]:
#  importar modulos
import pandas as pd
import requests
import pyodbc
# from openpyxl.workbook import Workbook
# from openpyxl import load_workbook
# from openpyxl import Workbook
# from openpyxl.worksheet.table import Table, TableStyleInfo
import calendar
from datetime import date, datetime, timedelta
from striprtf.striprtf import rtf_to_text
import os
import re
import unidecode
import io
import yaml
from unicodedata import normalize
from bs4 import BeautifulSoup
import json
import math
from glob import glob
import numpy as np
import sys
import spacy
from transformers import pipeline
import spacy_udpipe
# from datetime import datetime, date
import unicodedata
from pyspark.sql import functions as F

In [0]:
nlp = spacy.load("pt_core_news_sm")
doc = nlp("Testando o modelo em português.")
print([token.text for token in doc])

In [0]:
spacy_udpipe.download("pt-bosque")

####Variáveis de Ambiente

In [0]:
dbutils.widgets.text("id_projeto", "pulmao", "ID Projeto")
id_projeto = dbutils.widgets.get("id_projeto")
print("id_projeto:", id_projeto)

In [0]:
dbutils.widgets.text("environment", "dev", "Environment")
environment = dbutils.widgets.get("environment")
print("environment:", environment)

In [0]:
environment_tbl = "" if environment in ["hml", "prd"] else f"{environment}_"
print("environment_tbl:", environment_tbl)

In [0]:
# dbutils.widgets.text("catalog", "diamond_pulmao", "Catalog")
# catalog_name = dbutils.widgets.get("catalog")
# print(f"catalog_name: {catalog_name}")

In [0]:
# dbutils.widgets.text("schema", "pulmao", "Schema")
# schema_name = dbutils.widgets.get("schema")
# print(f"schema_name: {schema_name}")

In [0]:
# dbutils.widgets.text("work_catalog", "diamond_pulmao", "Work Catalog")
# work_catalog_name = dbutils.widgets.get("work_catalog")
# print(f"work_catalog_name: {work_catalog_name}")

In [0]:
# dbutils.widgets.text("work_schema", "workarea", "Work Schema")
# work_schema_name = dbutils.widgets.get("work_schema")
# print(f"work_schema_name: {work_schema_name}")

In [0]:
dbutils.widgets.text("data_execucao_modelo", "", "Data Execução Modelo")
data_execucao_modelo = dbutils.widgets.get("data_execucao_modelo")
if data_execucao_modelo == "":
    data_execucao_modelo = datetime.now().strftime("%Y-%m-%d")

In [0]:
INPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_pulmao"
OUTPUT_TABLENAME = f"{environment_tbl}tb_diamond_mod_pulmao_saida"
OUTPUT_TABLENAME_SF = f"{environment_tbl}tb_diamond_mod_pulmao_saida_salesforce"
DESCRIPTO_TABLENAME = f"{environment_tbl}tb_diamond_mod_pulmao_saida_descripto"
METRICS_TABLENAME = f"{environment_tbl}tb_diamond_mod_pulmao_metricas"
CATALOG = "ia"

In [0]:
print('Environment:    ', environment_tbl)
print('Tabela entrada: ', INPUT_TABLENAME)
print('Tabela destino: ', OUTPUT_TABLENAME)
print('Tabela destino: ', OUTPUT_TABLENAME_SF)
print('Tabela descrip: ', DESCRIPTO_TABLENAME)
print('Tabela metricas:', METRICS_TABLENAME)
print('catalogo:       ', CATALOG)
print('Data Referencia:', data_execucao_modelo)

In [0]:
# %sql
# DELETE FROM ia.dev_tb_diamond_mod_pulmao_metricas
# -- WHERE cast(dataExecucaoModelo as date) = '2026-01-13'
# WHERE cast(dia as date) = '2026-01-13'

In [0]:
# %sql
# select *
# from ia.dev_tb_diamond_mod_pulmao
# where cast(dataExecucaoModelo as date) = '2026-01-13'

####Arquivos csv descritores

In [0]:
def converte_df_antigo_para_novo(df_antigo):
    linhas_novas = []
    
    for _, row in df_antigo.iterrows():
        classe = str(row.iloc[0]).strip()
        if not classe or classe.lower() == 'nan':
            continue
        
        # valor base: pego da primeira variação (coluna 1)
        valor_base = str(row.iloc[1]).strip() if len(row) > 1 else ""
        
        # todas as variações: das colunas 1..N
        variacoes = []
        for val in row.iloc[1:]:
            val = str(val).strip()
            if val and val.lower() != 'nan':
                variacoes.append(val)
        
        # nova linha: classe, valor_base, lista (em string)
        linhas_novas.append([classe, valor_base, str(variacoes)])
    
    df_novo = pd.DataFrame(linhas_novas, columns=["_c0", "_c1", "_c2"])
    return df_novo

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/classificacao-e-localizacao-ia-tipoPT.csv")
df_classificacao_localizacao_tipoPT = df.toPandas()
df_classificacao_localizacao_tipoPT = converte_df_antigo_para_novo(df_classificacao_localizacao_tipoPT)
df_classificacao_localizacao_tipoPT.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/classificacao-e-localizacao-ia-riscopt.csv")
df_classificacao_localizacao_riscoPT = df.toPandas()
df_classificacao_localizacao_riscoPT = converte_df_antigo_para_novo(df_classificacao_localizacao_riscoPT)
df_classificacao_localizacao_riscoPT.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-descritores-fora-pulmao.csv")
descritores_ia_descritores_fora_pulmao = df.toPandas()
descritores_ia_descritores_fora_pulmao = converte_df_antigo_para_novo(descritores_ia_descritores_fora_pulmao)
descritores_ia_descritores_fora_pulmao.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-descritores-estruturais-prim2.csv")
descritores_ia_descritores_estruturais_prim = df.toPandas()
descritores_ia_descritores_estruturais_prim = converte_df_antigo_para_novo(descritores_ia_descritores_estruturais_prim)
descritores_ia_descritores_estruturais_prim.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-procedimento.csv")
descritores_ia_descritores_procedimento = df.toPandas()
descritores_ia_descritores_procedimento = converte_df_antigo_para_novo(descritores_ia_descritores_procedimento)
descritores_ia_descritores_procedimento.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-qualitativos-indet.csv")
descritores_ia_qualitativos_indet = df.toPandas()
descritores_ia_qualitativos_indet = converte_df_antigo_para_novo(descritores_ia_qualitativos_indet)
descritores_ia_qualitativos_indet.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-estruturais-sec.csv")
descritores_ia_estruturais_sec = df.toPandas()
descritores_ia_estruturais_sec = converte_df_antigo_para_novo(descritores_ia_estruturais_sec)
descritores_ia_estruturais_sec.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-comparativos.csv")
descritores_ia_descritores_comparativos = df.toPandas()
descritores_ia_descritores_comparativos = converte_df_antigo_para_novo(descritores_ia_descritores_comparativos)
descritores_ia_descritores_comparativos.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-frasepadrao.csv")
descritores_ia_frasespadrao = df.toPandas()
descritores_ia_frasespadrao = converte_df_antigo_para_novo(descritores_ia_frasespadrao)
descritores_ia_frasespadrao.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-qualitativos-m-ou-s.csv")
descritores_ia_descritores_qualitativo_m_ou_s = df.toPandas()
descritores_ia_descritores_qualitativo_m_ou_s = converte_df_antigo_para_novo(descritores_ia_descritores_qualitativo_m_ou_s)
descritores_ia_descritores_qualitativo_m_ou_s.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-qualitativo-benign.csv")
descritores_ia_descritores_qualitativo_benign = df.toPandas()
descritores_ia_descritores_qualitativo_benign = converte_df_antigo_para_novo(descritores_ia_descritores_qualitativo_benign)
descritores_ia_descritores_qualitativo_benign.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "false") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/Descritores-ia-qualitativo-de-con.csv")
descritores_ia_descritores_qualitativo_de_con = df.toPandas()
descritores_ia_descritores_qualitativo_de_con = converte_df_antigo_para_novo(descritores_ia_descritores_qualitativo_de_con)
descritores_ia_descritores_qualitativo_de_con.shape

In [0]:
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiline", "true") \
    .load(f"/mnt/trusted/datalake/ia/projetos/{id_projeto}/config/{environment}/arquivo_depara_unidades_salesforce.csv")
depara_unidades = df.toPandas()
depara_unidades.shape

####Funções

In [0]:
## FUNCIONA
def leDicionario(arquivo_ou_df, aba=None):
    if isinstance(arquivo_ou_df, str):
        if aba is None:
            raise ValueError("Quando 'arquivo_ou_df' é string, 'aba' deve ser fornecida.")
        df = pd.read_excel(arquivo_ou_df, sheet_name=aba)
    elif isinstance(arquivo_ou_df, pd.DataFrame):
        df = arquivo_ou_df
    else:
        raise TypeError("arquivo_ou_df deve ser string (caminho) ou pd.DataFrame")

    # Força reset de índice e colunas numéricas
    df = df.reset_index(drop=True)
    df.columns = [f'_c{i}' for i in range(len(df.columns))]

    dict1 = {}
    dict2 = {}

    for _, row in df.iterrows():
        if len(row) < 2:
            continue
        chave = row['_c0']
        valor = row['_c1']
        variaacoes = []

        if len(row) >= 3 and pd.notna(row['_c2']):
            try:
                variaacoes = eval(str(row['_c2']))
            except:
                variaacoes = [str(row['_c2'])]

        if pd.notna(chave) and pd.notna(valor):
            dict1[chave] = valor
            dict2[chave] = variaacoes

    return dict1, dict2

In [0]:
def criaPadroes(dictPadroes, tipo = 2):
   patternsBR = [] 
   for key in dictPadroes.keys():
       if tipo == 1:
          dict = criaPadrao(key, dictPadroes[key])   
          patternsBR.append(dict)
       else:
          padroes = criaPadrao2(key, dictPadroes[key])   
          patternsBR.extend(padroes)
   return patternsBR 

In [0]:
# esta é a funcao correta
def criaPadrao2(padrao, lista):
    padroes = []
    for palavra in lista:
        dictp = {}
        dictp['label'] = padrao
        dictp['pattern'] = []
        if ' ' in palavra:
           tokens = palavra.split(' ')
        else:
           tokens = [palavra] 
        for token in tokens:
            patterndict = {}
            if not '%' in token and not '.' in token:  
               patterndict['LOWER'] = token
            else:
               patterndict['LOWER'] = {'REGEX' : '(\\b'+ token.replace('%', '(\\w*)') + ')'} ##<< flexibiliza a captura do achado descritor
            dictp['pattern'].append(patterndict)
        padroes.append(dictp)
    
    return padroes

In [0]:
def preparaNER(padroes):
        config = {
           "phrase_matcher_attr": "LOWER", ##<<<<< alterado de None para LOWER
           "validate": True,
           "overwrite_ents": True,
           "ent_id_sep": "||",
        }
    
        # nlp = spacy.load("pt_core_news_sm")
        nlp = spacy.load("pt_core_news_lg")
        ruler = nlp.add_pipe("entity_ruler", config=config)

        ruler.add_patterns(padroes)
        return nlp

In [0]:
def configuraRuler(nlp):
    config = {
           "phrase_matcher_attr": "LOWER", ##<<<<< alterado de None para LOWER
           "validate": True,
           "overwrite_ents": True,
           "ent_id_sep": "||",
    }
        
    ruler = nlp.add_pipe("entity_ruler", config=config)
    return ruler

In [0]:
def padroesDefault():
    patternsBR = [
                {"label": "DATE", "pattern": [{"SHAPE": "dd/dd/dd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "dd/dd/dddd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "d/dd/dddd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "d/d/dddd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "d/d/dd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "dd/dddd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "dd/dd"}]},

                {"label": "DATE", "pattern": [{"SHAPE": "dd-dd-dd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "dd-dd-dddd"}]},

                {"label": "DATE", "pattern": [{"SHAPE": "dd.dd.dd"}]},
                {"label": "DATE", "pattern": [{"SHAPE": "dd.dd.dddd"}]},

                {"label": "TIME", "pattern": [{"SHAPE": "dd:dd"}]},
                {"label": "TAMANHO", "pattern": 
                   [
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": 'x'},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": 'x'},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": {"IN": ["cm", "centimetros", "centimetro", "mm", "milímetro", "milimetros"]}}
                   ]
                },
                {"label": "TAMANHO", "pattern": 
                   [
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": 'e'},
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": {"IN": ["cm", "centimetros", "centimetro", "mm", "milímetro", "milimetros"]}}
                   ]
                },
                {"label": "TAMANHO", "pattern": 
                   [
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": 'x'},
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "?"},
                     {"LOWER": {"IN": ["cm", "centimetros", "centimetro", "mm", "milímetro", "milimetros"]}}
                   ]
                },
                {"label": "TAMANHO", "pattern": 
                   [
                     {"LIKE_NUM": True},
                     {"IS_SPACE": True, "OP": "*"},
                     {"LOWER": {"IN": ["cm", "centimetros", "centimetro", "mm", "milímetro", "milimetros"]}}
                   ]
                },
            ] 

    return patternsBR

In [0]:
def criaPadraoConjunto(padrao, lista1, lista2):
    lista = [a + ' ' + b for a, b in itertools.product(lista1, lista2)];
    return criaPadrao2(padrao, lista);  

In [0]:
def removeAcentos(texto):
    texto = texto.lower()
    texto = normalize('NFKD', texto).encode('ASCII', 'ignore').decode('ASCII')
    return texto

In [0]:
def sentencizer(texto):
    linhas = texto.split('\n')
    sentencas = []
    for linha in linhas:
        subfrases = linha.split(';')
        for subfrase in subfrases:
           sents = tokenize.sent_tokenize(subfrase)
           sentencas.extend(sents)
    return sentencas

In [0]:
import itertools
from nltk import tokenize

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')  # só na primeira vez

# tokens = word_tokenize(texto, language='portuguese')

In [0]:
def doc2text(doc, tipo=1):
    result = ''
    if tipo == 0:
       for token in doc:
           result = result + token.text + token.whitespace_
       return result
    elif tipo == 1:
       for token in doc:
           result = result + ' ' + token.text
       return result[1:]

In [0]:
def pegaTamanho(ie):
    tam = 0
    try: 
       for nodulo in ie.nodulos:
           if tam < nodulo.tamanhoMaximo():
              tam = nodulo.tamanhoMaximo()
    except:
       tam = 0
    return tam  

In [0]:
def identificaEntidades(nlpExterno, texto):
    texto = removeAcentos(texto) 
    doc = nlpExterno(texto)
    ents = []
    for ent in doc.ents:
        if ent.label_ not in ['PER', 'LOC', 'MISC', 'ORG']:
           ents.append(ent.label_)
    ents = list(set(ents))
    return ents

In [0]:
def juntaLista(lista):
    result = ''
    for palavra in lista:
        if result == '':
           result = palavra
        else:   
           result = result + ' / '+ palavra
    return result

In [0]:
def string2Date(texto: str) -> date:
    t = str(texto).strip()

    # 1) Apenas dígitos? (ex.: '20251010' ou '10102025')
    if t.isdigit() and len(t) == 8:
        # tenta YYYYMMDD
        try:
            return datetime.strptime(t, "%Y%m%d").date()
        except ValueError:
            pass
        # tenta DDMMYYYY
        try:
            return datetime.strptime(t, "%d%m%Y").date()
        except ValueError:
            pass

    # 2) Formatos comuns com separadores
    formatos = (
        "%d/%m/%Y", "%Y-%m-%d", "%d-%m-%Y", "%d.%m.%Y",
        "%d/%m/%y",  "%d-%m-%y",  "%d.%m.%y"   # aceita ano 2 dígitos
    )
    for fmt in formatos:
        try:
            return datetime.strptime(t, fmt).date()
        except ValueError:
            continue

    raise ValueError(f"Formato de data não reconhecido: {texto}")

def converteData(data) -> date:
    return string2Date(data) if isinstance(data, str) else data

In [0]:
def normaliza_nodulo(texto: str) -> str:
    """
    Transforma padrões como 'nódulo de 0,7 cm' em 'nódulo medindo 0,7 cm',
    para que o parser de tamanho reconheça corretamente.
    """
    # regex: pega "nódulo de <número + unidade>"
    padrao = re.compile(r'(\bn[óo]dulo(?:s)?)(\s+de\s+)([\d\.,]+\s*(?:cm|mm))', re.IGNORECASE)
    return padrao.sub(r'\1 medindo \3', texto)

In [0]:
def html2Text(texto: str) -> str:
    if not isinstance(texto, str):
        return texto
    soup = BeautifulSoup(texto, features="lxml")
    text = soup.get_text()
    return text.replace('\n\n\n', '\n').replace(u'\x13', '').replace(u'\xa0', ' ').replace(u'\x0b', '')

In [0]:
def retiraIlegais(texto: str) -> str:
    """
    Remove caracteres de controle, invisíveis e não imprimíveis
    (ex.: \x00–\x1F, \x7F–\x9F, \uffff, NBSP, etc.).
    Retorna o texto limpo e seguro para regex/spaCy.
    """
    if not isinstance(texto, str):
        return texto

    # Remove caracteres de controle e não imprimíveis
    pattern = re.compile(r'[\x00-\x09\x0b\x0c\x0e-\x1f\x7f-\x9f\uffff]')
    texto = pattern.sub('', texto)

    # Remove espaços não separáveis (NBSP, etc.)
    texto = texto.replace('\xa0', ' ').replace('\u200b', ' ')

    # Padroniza múltiplos espaços
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto

In [0]:
def normaliza_laudo(texto: str) -> str:
    """
    Normaliza o laudo para facilitar os padrões (regex):
    - Insere espaço após vírgula/ponto quando colados em letras.
    - Converte vírgula decimal para ponto (1,0 -> 1.0).
    - Padroniza ' x ' em medidas (x com espaços em volta).
    - Corrige colagens comuns (onódulo -> o nódulo; 'lobulado,com' -> 'lobulado, com').
    - Mantém acentuação.
    """
    if not texto:
        return texto

    t = texto

    # 1) vírgula/ponto colados em letras (ex.: "lobulado,com" -> "lobulado, com")
    t = re.sub(r'([A-Za-zÁ-ü])([,.;:])([A-Za-zÁ-ü])', r'\1\2 \3', t)

    # 2) vírgula decimal -> ponto (ex.: 1,0 -> 1.0 ; 0,8 -> 0.8)
    t = re.sub(r'(?<=\d),(?=\d)', '.', t)

    # 3) padroniza "x" de medidas (garante espaços ao redor do x)
    # t = re.sub(r'\s*[xX×]\s*', ' x ', t)
    t = re.sub(r'(\d+[,\.]?\d*)\s*[xX×]\s*(\d+[,\.]?\d*)', r'\1 x \2', t)

    # 4) pequenos fixes de colagens/faltas de espaço que atrapalham gatilhos
    # "onódulo" geralmente é "o nódulo"
    t = re.sub(r'\bonódulo\b', 'o nódulo', t, flags=re.IGNORECASE)

    # se houver "nódulo" colado em algo anterior (ex.: "umonódulo") tenta separar
    #t = re.sub(r'([a-zá-ü])n[óo]dulo', r'\1 nódulo', t, flags=re.IGNORECASE)
    #t = re.sub(r'\b(n[óo]dulo)(?=de\b)', r'\1 ', t, flags=re.IGNORECASE)  # insere espaço: "nódulode" -> "nódulo de"

    # 5) padroniza múltiplos espaços
    t = re.sub(r'\s+', ' ', t).strip()

    fixes = {
        r'\bdomeio\b': 'do meio',
        r'\bapicoposteiror\b': 'apicoposterior',
        r'\bposteiror\b': 'posterior',
        r'\bbilateraiscom\b': 'bilaterais com',
        r'\bTraqueiapérvia\b': 'Traqueia pérvia',
        r'\bbronquioloectasia\b': 'bronquiolectasia',   
        r'\bsemi-sólida\b': 'semissólida',
        r'\bonóedulo\b': 'o nódulo',
        r'\bnódulode\b': 'nódulo de',
        r'\b:nódulo\b': 'nódulo',          
    }
    for patt, repl in fixes.items():
        t = re.sub(patt, repl, t, flags=re.IGNORECASE)

    # 6) espaços múltiplos -> um espaço
    t = re.sub(r'\s+', ' ', t).strip()

    return t

In [0]:
PADRAO_MEDINDO = re.compile(
    r'\b(?:medindo|com|inferiores a|menores)'                                
    r'(?:\s+(?:cerca\s+de|aprox\.?|aproximadamente|que|do que|que fizeram|até|ate|em\s+torno\s+de))?'  # complementos
    r'\s+' 
    r'(?P<a>\d+(?:[\.,]\d+)?)'                          
    r'(?:\s*[x×]\s*(?P<b>\d+(?:[\.,]\d+)?))?'           
    r'(?:\s*[x×]\s*(?P<c>\d+(?:[\.,]\d+)?))?'           
    r'\s*(?P<unidade>mm|cm)\b',
    flags=re.IGNORECASE
)

# 2.2) padrão "nódulo ... de X mm/cm" ou "nódulo ... X mm/cm"
PADRAO_NODULO_DE_MEDIDA = re.compile(
    r'\bn[óo]dulo(?:s)?\b[^.]{0,120}?'
    r'(?:\bde\b\s*)?'
    r'(?P<valor>\d+(?:\.\d+)?)\s*(?P<unidade>mm|cm)\b',
    flags=re.IGNORECASE
)

def _to_mm(valor: float, unidade: str) -> float:
    """Converte para milímetros."""
    return valor * 10.0 if unidade.lower() == 'cm' else valor

def extrai_medidas_generico(frase: str):
    """
    Tenta extrair medidas em ordem:
    1) 'medindo (cerca de) A x B (x C) cm/mm' -> avalia TODAS as ocorrências e retorna o MAIOR eixo (mm)
    2) 'nódulo (de) X mm/cm' -> avalia TODAS as ocorrências e retorna o MAIOR valor (mm)
    Ignora medidas antigas dentro de parênteses que mencionem 'anterior/prévio/comparad'.
    Retorna dict com 'texto' (concat das medidas atuais usadas), 'tamanho_mm' (maior em mm) e 'unidade_base' ('mm').
    """
    if not frase:
        return None

    txt = str(frase)

    # 1) remover blocos ENTRE parênteses que indiquem exame anterior/prévio/comparativo
    #    (evita contaminar o cálculo com medidas antigas)
    txt = re.sub(
        r"\([^)]*(anterior|pr[eé]vio|previo|comparad)[^)]*\)",
        "",
        txt,
        flags=re.IGNORECASE
    )

    # 2) normalizações para robustez
    txt = txt.replace("×", "x")        # multiplicador
    txt = txt.replace(",", ".")        # decimais com vírgula

    valores_mm = []
    trechos_usados = []

    # 3) pegar TODAS as ocorrências do padrão "medindo ..."
    for m in PADRAO_MEDINDO.finditer(txt):
        unidade = m.group('unidade')
        nums = [m.group('a'), m.group('b'), m.group('c')]
        nums = [float(x) for x in nums if x]
        if not nums:
            continue
        maior = max(nums)
        valores_mm.append(_to_mm(maior, unidade))
        trechos_usados.append(m.group(0))

    # 4) se não achou "medindo ...", tentar "nódulo ... de X mm/cm" (todas as ocorrências)
    if not valores_mm:
        for m2 in PADRAO_NODULO_DE_MEDIDA.finditer(txt):
            try:
                valor = float(m2.group('valor'))
            except:
                continue
            unidade = m2.group('unidade')
            valores_mm.append(_to_mm(valor, unidade))
            trechos_usados.append(m2.group(0))

    if not valores_mm:
        return None

    tamanho_max_mm = max(valores_mm)
    texto_concat = " / ".join(trechos_usados) if trechos_usados else None

    return {
        'texto': texto_concat,
        'tamanho_mm': tamanho_max_mm,
        'unidade_base': 'mm'  # base do retorno (tamanho_mm)
    }

In [0]:
## se houver para o mesmo id mais de uma classificação no mesmo dia com o mesmo laudo aqui é concatenado
## se houver mais de um laudo no mesmo dia mas com contexto diferente não é concatenado
## se localizar mais de um nodulo no mesmo laudo ele mantem uma unica linha e registra ambas as capturas separadas por //-//
def agrupar_por_id_concatenando(df: pd.DataFrame) -> pd.DataFrame:
    """
    Retorna um DataFrame com 1 linha por 'id', concatenando (com ' / ')
    os valores das colunas-alvo, sem duplicar e mantendo a ordem de aparição.
    """

    colunas_alvo = [
        'frase_original',
        'frase_processada',
        'localizacao',
        'regiao',
        'medidas',
        'medidas_limpa',
        'tamanho_mm',
        'descricao',
        'classificacoes',
        'riscos',
        'categorizacao',
    ]

    # --- Funções auxiliares ---
    def _is_vazio(x):
        if x is None:
            return True
        if isinstance(x, float) and math.isnan(x):
            return True
        if isinstance(x, str) and x.strip() == "":
            return True
        return False

    def _to_text(x):
        # Formata números sem .0 quando forem inteiros
        if isinstance(x, (int, float)) and not isinstance(x, bool):
            if isinstance(x, float) and math.isnan(x):
                return None
            if float(x).is_integer():
                return str(int(x))
            else:
                return f"{float(x):.1f}"
        # Converte listas/tuplas para itens individuais separados
        if isinstance(x, (list, tuple, set)):
            # transforma elementos em texto e junta com espaço (serão deduplicados depois)
            return " ".join([_to_text(v) for v in x if not _is_vazio(v)]) or None
        # Qualquer outro vira string
        s = str(x)
        return s if s.strip() else None

    def _concat_unicos_ordenado(series: pd.Series) -> str:
        vistos = set()
        ordered = []
        for v in series:
            if _is_vazio(v):
                continue
            texto = _to_text(v)
            if texto is None or texto.strip() == "":
                continue
            # Divide por " / " caso já venha composto, preservando ordem
            partes = [p.strip() for p in texto.split("//-//") if p.strip() != ""]
            for p in partes:
                if p not in vistos:
                    vistos.add(p)
                    ordered.append(p)
        return " //-// ".join(ordered)

    # Garante que todas as colunas alvo existam (se faltar, cria vazia)
    for c in colunas_alvo:
        if c not in df.columns:
            df[c] = None

    # Agrupa por 'id' e aplica agregação personalizada
    agrupado = (
        # df.groupby(['exm_laudo_dataliber', 'laudo_texto', 'exm_an', 'redcap_repeat_instance'], sort=False)
        # df.groupby(['id', 'exm_laudo_dataliber', 'laudo_texto'], sort=False)

        df.groupby('exm_an', sort=False) ##<< usar com lake

          .agg(_concat_unicos_ordenado)
          .reset_index()
    )

    return agrupado


In [0]:
def limpa_descritores(df: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada id, mantém apenas a primeira ocorrência de exm_descritores_laudo
    quando houver valores repetidos. As demais recebem NaN.
    """
    df = df.copy()
    df["exm_descritores_laudo"] = df.groupby("record_id")["exm_descritores_laudo"] \
        .transform(lambda x: x.where(~x.duplicated(), None))
    return df

In [0]:
def ids_nao_presentes(df_origem: pd.DataFrame, df_destino: pd.DataFrame,
                      col_origem: str = "an", col_destino: str = "exm_an") -> pd.Series:
    """
    Retorna os valores da coluna col_origem (df_origem)
    que não estão presentes na coluna col_destino (df_destino).
    """
    return df_origem[~df_origem[col_origem].isin(df_destino[col_destino])][col_origem].unique()

def ids_presentes(df_origem: pd.DataFrame, df_destino: pd.DataFrame,
                      col_origem: str = "an", col_destino: str = "exm_an") -> pd.Series:
    """
    Retorna os valores da coluna col_origem (df_origem)
    que não estão presentes na coluna col_destino (df_destino).
    """
    return df_origem[df_origem[col_origem].isin(df_destino[col_destino])][col_origem].unique()

In [0]:
def remove_acentos(texto):
    if pd.isna(texto):
        return texto
    return ''.join(c for c in unicodedata.normalize('NFD', texto)
                   if unicodedata.category(c) != 'Mn')

In [0]:
def amostra(df, total_amostras=510):
    col_data = 'data_ref'  
    total_amostras = total_amostras

    dias_unicos = df[col_data].nunique()
    amostras_por_dia = total_amostras // dias_unicos

    amostra = df.groupby(col_data, group_keys=False).apply(
        lambda x: x.sample(
            n=min(amostras_por_dia, len(x)),
            random_state=42
        )
    )
    amostra = amostra.head(500)
    return amostra.reset_index(drop=True)

In [0]:
def limpa_colunas_texto(df, colunas):
    df = df.copy()

    for col in colunas:
        if col in df.columns:
            df[col] = (
                df[col]
                .fillna('')  # garante que valores nulos não quebrem
                .astype(str)  # força string
                .str.replace(r'\s+', ' ', regex=True)  # substitui múltiplos espaços
                .str.replace(r'\n', ' ', regex=True)   # remove quebras de linha
                .str.replace(r'[\\/"]', '', regex=True) # remove / \ "
                .str.replace("'", '', regex=False)      # remove '
                .str.strip()                            # remove espaços no início e fim
            )

            # volta os NaN nos casos que ficaram vazios
            df.loc[df[col] == '', col] = None

    return df

In [0]:
def envia_csv(df, nome_arquivo):
    csv_import = spark.createDataFrame(df)
    # print(type(df_pulmao_csv_import))
    csv_import.repartition(1).write.option("header",True).csv(f"/mnt/temp/sandbox/leandro_neri/pulmao/amostras_laudos/{nome_arquivo}.csv")
    print(f'Salvou {nome_arquivo}.csv')

    return csv_import

In [0]:
def normalizar_nome(texto):
    if pd.isna(texto):
        return None
    
    texto = unidecode.unidecode(texto.upper())
    
    # remove palavras comuns
    texto = re.sub(
        r'\b(HOSPITAL|SAO|SAO LUIZ|LUIZ|REDE|UNIDADE|E MATERNIDADE)\b',
        '',
        texto
    )
    
    # remove espaços extras
    texto = re.sub(r'\s+', ' ', texto).strip()
    
    return texto

####Classes

In [0]:
class infoCore:
    def __init__(self, ie):
        self.ie = ie
        self.tamanho = ""
        self.localizacao = ""
        self.tamanhoMaximo = 0.0

    # ##original
    # def setaTamanho(self, tamanho):
    #     # Tenta extrair número e unidade
    #     import re
    #     self.tamanho = tamanho
    #     # Extrai número (ex: "6mm" → 6.0)
    #     numeros = re.findall(r"[\d\.,]+", tamanho)
    #     if numeros:
    #         try:
    #             valor = float(numeros[0].replace(",", "."))
    #             # Se for cm, converte para mm
    #             if "cm" in tamanho.lower():
    #                 valor *= 10
    #             self.tamanhoMaximo = valor
    #             return True
    #         except:
    #             pass
    #     return False

    # # adicionado no lugar da função acima
    # def setaTamanho(self, tamanho):
    #     import re
    #     self.tamanho = tamanho
    #     numeros = re.findall(r"[\d\.,]+", tamanho)
    #     if numeros:
    #         try:
    #             valor = float(numeros[0].replace(",", "."))
    #             if "cm" in tamanho.lower():
    #                 valor *= 10
    #             self.tamanhoMaximo = valor
    #             return True
    #         except Exception as e:
    #             print(f"[ERRO setaTamanho] {e}")
    #             pass
    #     return False

    ## adicionado no lugar da funcao acima
    def setaTamanho(self, tamanho: str) -> bool:
        """
        Estratégia:
        1) Procurar padrões com 'medindo|mede|medem|medida de' (prioridade alta).
        2) Se não achar, remover parênteses com 'média' e procurar medidas gerais.
        3) Fallback: primeira medida geral se nada acima funcionar.
        Converte cm->mm e grava em self.tamanhoMaximo.
        """
        import re

        def _to_mm(valor_str: str, unidade: str) -> float:
            # Normaliza decimal com vírgula
            v = float(valor_str.replace(",", "."))
            return v * 10.0 if unidade.lower().startswith("cm") else v

        # Guarda string original
        self.tamanho = tamanho if isinstance(tamanho, str) else str(tamanho)
        texto = self.tamanho.lower()

        # -------------------------------
        # 1) PRIORIDADE: frases com verbo de medida (medindo/mede/medem/medida de)
        #    Captura: "medindo 0,8 cm", "mede 6 mm", "medida de 0,5 cm", etc.
        # -------------------------------
        ##original 02 inicio
        padrao_prioritario = re.compile(
            r"(?:medindo|mede|medem|medida\s+de)\s+([\d\.,]+)\s*(cm|mm)",
            flags=re.IGNORECASE
        )
        ##original02 fim
        # padrao_prioritario = re.compile(
        #     r"(?:medindo|mede|medem|medida\s+de)"
        #     r"(?:\s+(?:cerca\s+de|aprox\.?|aproximadamente|até|em\s+torno\s+de))?"
        #     r"\s+([\d\.,]+)\s*(cm|mm)",
        #     flags=re.IGNORECASE
        # ) 
        m = padrao_prioritario.search(texto)
        if m:
            try:
                valor_mm = _to_mm(m.group(1), m.group(2))
                self.tamanhoMaximo = valor_mm
                return True
            except Exception as e:
                print(f"[ERRO setaTamanho/prioritario] {e}")

        # -------------------------------
        # 2) Remover parênteses que contêm 'média' para evitar capturar medidas históricas,
        #    ex.: "(média 0,6 cm)".
        # -------------------------------
        texto_sem_media_par = re.sub(r"\([^)]*m[eé]dia[^)]*\)", " ", texto, flags=re.IGNORECASE)

        # Tenta novamente padrão prioritário após limpar '(média ...)'
        m2 = padrao_prioritario.search(texto_sem_media_par)
        if m2:
            try:
                valor_mm = _to_mm(m2.group(1), m2.group(2))
                self.tamanhoMaximo = valor_mm
                return True
            except Exception as e:
                print(f"[ERRO setaTamanho/prioritario_2] {e}")

        # -------------------------------
        # 3) Fallback: qualquer número + unidade (cm|mm) no texto sem '(média ...)'
        # -------------------------------
        padrao_geral = re.compile(r"([\d\.,]+)\s*(cm|mm)", flags=re.IGNORECASE)
        m3 = padrao_geral.search(texto_sem_media_par)
        if m3:
            try:
                valor_mm = _to_mm(m3.group(1), m3.group(2))
                self.tamanhoMaximo = valor_mm
                return True
            except Exception as e:
                print(f"[ERRO setaTamanho/geral] {e}")

                # -------------------------------
        # # 3) Fallback: qualquer número + unidade (cm|mm) no texto sem '(média ...)'
        # #    MAS captura o contexto completo (ex: "menores que 0,3 cm")
        # # -------------------------------
        # # Padrões de contexto: "menores que X", "até X", "maiores que X", etc.
        # padroes_contexto = [
        #     r'(menores? que|até|maiores? que|superiores? a|inferiores? a)\s*([\d\.,]+)\s*(cm|mm)',
        #     r'([\d\.,]+)\s*(cm|mm)\s*(ou menos|ou mais|no máximo|no mínimo)'
        # ]
        
        # for padrao_str in padroes_contexto:
        #     padrao = re.compile(padrao_str, flags=re.IGNORECASE)
        #     m3 = padrao.search(texto_sem_media_par)
        #     if m3:
        #         try:
        #             # Captura a frase completa (ex: "menores que 0,3 cm")
        #             self.tamanho = m3.group(0)
        #             valor_mm = _to_mm(m3.group(2) if len(m3.groups()) >= 3 else m3.group(1), 
        #                             m3.group(3) if len(m3.groups()) >= 3 else m3.group(2))
        #             self.tamanhoMaximo = valor_mm
        #             return True
        #         except Exception as e:
        #             print(f"[ERRO setaTamanho/contexto] {e}")
        
        # # Se não encontrou contexto, busca número + unidade normalmente
        # padrao_geral = re.compile(r"([\d\.,]+)\s*(cm|mm)", flags=re.IGNORECASE)
        # m3 = padrao_geral.search(texto_sem_media_par)
        # if m3:
        #     try:
        #         valor_mm = _to_mm(m3.group(1), m3.group(2))
        #         self.tamanhoMaximo = valor_mm
        #         return True
        #     except Exception as e:
        #         print(f"[ERRO setaTamanho/geral] {e}")
                # -------------------------------
        # 3) Fallback: captura medidas com contexto ("menores que", "até", etc) OU medidas gerais
        # -------------------------------
        # padroes_contexto = [
        #     r'(menores? que|até|maiores? que|superiores? a|inferiores? a)\s*([\d\.,]+)\s*(cm|mm)',
        #     r'([\d\.,]+)\s*(cm|mm)\s*(ou menos|ou mais|no máximo|no mínimo|medindo)'
        # ]
        ## para corrigir um caso "menores que fizeram 0,6cm" id 20250703536084
        padroes_contexto = [
            r'(menores?\s+que|inferiores?\s+a|maiores?\s+que|superiores?\s+a|até)\s+(?:\w+\s+){0,3}?([\d\.,]+)\s*(cm|mm)',
            r'([\d\.,]+)\s*(cm|mm)\s*(ou menos|ou mais|no máximo|no mínimo|medindo)'
        ]
        for padrao_str in padroes_contexto:
            padrao = re.compile(padrao_str, flags=re.IGNORECASE)
            m3 = padrao.search(texto_sem_media_par)
            if m3:
                try:
                    # Captura a frase completa (ex: "menores que 0,3 cm")
                    self.tamanho = m3.group(0)
                    
                    # Extrai número e unidade
                    if len(m3.groups()) >= 3:
                        numero_str = m3.group(2)
                        unidade = m3.group(3)
                    else:
                        numero_str = m3.group(1)
                        unidade = m3.group(2)
                    
                    # Substitui vírgula por ponto ANTES de converter
                    numero_str = numero_str.replace(',', '.')
                    valor_mm = _to_mm(numero_str, unidade)
                    
                    self.tamanhoMaximo = valor_mm
                    return True
                except Exception as e:
                    print(f"[ERRO setaTamanho/contexto] {e}")
        
        # Fallback: qualquer número + unidade (cm|mm) no texto sem '(média ...)'
        padrao_geral = re.compile(r"([\d\.,]+)\s*(cm|mm)", flags=re.IGNORECASE)
        m3 = padrao_geral.search(texto_sem_media_par)
        if m3:
            try:
                # Substitui vírgula por ponto
                numero_str = m3.group(1).replace(',', '.')
                valor_mm = _to_mm(numero_str, m3.group(2))
                self.tamanhoMaximo = valor_mm
                return True
            except Exception as e:
                print(f"[ERRO setaTamanho/geral] {e}")

        # -------------------------------
        # 4) Último recurso: qualquer número (sem unidade) → assume mm
        # -------------------------------
        m4 = re.search(r"([\d\.,]+)", texto_sem_media_par)
        if m4:
            try:
                valor_mm = float(m4.group(1).replace(",", "."))
                self.tamanhoMaximo = valor_mm
                return True
            except Exception as e:
                print(f"[ERRO setaTamanho/numero_simples] {e}")

        # Não achou nada válido
        return False
    
    def setaLocalizacao(self, localizacao):
        self.localizacao = localizacao
        return True

    def inicializado(self):
        return self.tamanho != "" or self.localizacao != ""

In [0]:
class InformationExtraction:
      def __init__(self, tipoParser = 2):
          print(60*'*')
          print(60*'*')
          print('Criando ie')
          print(60*'*')
          
          self.tipoParser = tipoParser
          self.sentence_model = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
          self.printOutput = []

          self.nlp = spacy.load("pt_core_news_sm")
          if self.tipoParser == 1:
             self.parser = Parser.load('/dadosDL/codigos/machinelearning/spacy/Turku-neural-parser-pipeline/UD_Portuguese-Bosque-master/supar/xlm-roberta/model')
          elif self.tipoParser == 2:
             self.nlp_udpipe = spacy_udpipe.load("pt-bosque")

          self.stopwords = []
          self.nodulos = []
    
          self.ruler = configuraRuler(self.nlp)

          patternsBR = padroesDefault()

          self.leDescritores()
          self.leFrasesPadrao()
          patternsBR.extend(self.padroesProcedimento)
          patternsBR.extend(self.padroesIndeterminados)
          patternsBR.extend(self.padroesNaoPulmao)
          patternsBR.extend(self.padroesNodulares)
          patternsBR.extend(self.padroesSecundarios)
          patternsBR.extend(self.padroesComparativos)
          patternsBR.extend(self.padroesComparativosClassif)
          patternsBR.extend(self.padroesQualitativos)
          patternsBR.extend(self.padroesConduta)

          self.descritoresDoPulmao = ['SEGMENTO APICAL', 'SEGMENTO APICOPOSTERIOR', 'SEGMENTO ANTERIOR', 'SEGMENTO POSTERIOR', 'SEGMENTO LATERAL', 'SEGMENTO MEDIAL', 'SEGMENTO SUPERIOR', 'SEGMENTO BASAL ANTEROMEDIAL', 'SEGMENTO BASAL MEDIAL', 'SEGMENTO BASAL LATERAL', 'SEGMENTO BASAL ANTERIOR', 'SEGMENTO BASAL POSTERIOR', 'LOBO SUPERIOR', 'LOBO MEDIO', 'LOBO INFERIOR', 'LINGULA', 'PULMAO DIREITO', 'PULMAO ESQUERDO', 'PULMOES']
          self.descritoresForaPulmao = ['VESICULA', 'BEXIGA', 'UTERO', 'TROMPAS', 'INTESTINO', 'APENDICE', 'OVARIO', 'PAREDE', 'ABDOME', 'ADRENAL', 'AORTA', 'MAMA', 'ESOFAGO', 'ARTICULACAO', 'RIM', 'FIGADO', 'MEDIASTINO', 'PANCREAS', 'BACO', 'TIREOIDE', 'INFRAMAMARIO', 'SUBDIAFRAGMATICA', 'ESTOMAGO', 'PERITONIO', 'COSTELA', 'CLAVICULA', 'UMERO', 'VERTEBRA', 'PROSTATA', 'PELVE']

          self.ruler.add_patterns(patternsBR)

          self.dicionarioCaracterizacoes = {'SURGIMENTO': ['novo nódulo em relação', 'nódulo não identificado']}  
          self.dicionarioCaracterizacoes.update({'PROGRESSAO': ['o nódulo cresceu', 'nódulo aumentou', 'nódulo expandiu', 'nódulo piorou']})  
          self.dicionarioCaracterizacoes.update({'ESTABILIDADE': ['nódulo se manteve', 'nódulo permanece inalterado', 'não se observam alterações no nódulo', 'não se observam modificações no nódulo', 'permanece semelhante o nódulo']})
          self.dicionarioCaracterizacoes.update({'REDUCAO': ['o nódulo diminuiu', 'o nódulo reduziu', 'nódulo melhorou']})
          self.dicionarioCaracterizacoes.update({'RESOLUCAO': ['não mais se observa o nódulo', 'nódulo desapareceu']})

          self.sentence_classifier = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli")

          self.debug = False
          self.noduloAtual = None

      ##>> expressao de captura 'nodulo', 'nodulos', 'micronodulos' 
      def descritoresNodulares(self):
        # tentativa de corrigir a captura do tamanhoa
          self.padroesNodulares= [{"label": "NODULO", "pattern": [{'LOWER': {'REGEX': '((\\w+)n[oó]dul(\\w+))'}}]}]
              # 2) NOVO: captura “nódulo”/“nódulos” como palavra isolada (singular/plural)
          self.padroesNodulares.extend([
              {"label": "NODULO",  "pattern": [{"TEXT": {"REGEX": r"(?i)\bn[oó]dulo\b"}}]},
              {"label": "NODULOS", "pattern": [{"TEXT": {"REGEX": r"(?i)\bn[oó]dulos\b"}}]},
          ])
        ## padrao anterior ((\\w+)[mn][ioó]d[uú]l(\\w+))
        #   self.padroesNodulares = [
        #       {
        #           "label": "NODULO", 
        #           "pattern": [
        #               {
        #                   'LOWER': {
        #                       'REGEX': r'^(micro)?n[oó]d[uú]l(os?|ar(es)?|ares?)$'  # ← Aceita "nodulo", "nódulo", "micronódulo", "micronodulo"
        #                   }
        #               }
        #           ]
        #       }
        #   ]
          ##>> descritores que complementam a identificacao, foca em caracteristicas nao apenas em 'nodulo'
          _, tempDict = leDicionario(descritores_ia_descritores_estruturais_prim)
          tempPatt = criaPadroes(tempDict, 2)
          self.padroesNodulares.extend(tempPatt)

      def leDescritores(self): 
        #   self.padroesProcedimento = criaPadroes(leDicionario('Descritores.xlsx', 'DESCRITORES PROCEDIMENTO')[1], 2)
          self.padroesProcedimento = criaPadroes(leDicionario(descritores_ia_descritores_procedimento)[1], 2)
        #   self.padroesNaoPulmao = criaPadroes(leDicionario('Descritores.xlsx', 'DESCRITORES FORA PULMAO')[1], 2)
          self.padroesNaoPulmao = criaPadroes(leDicionario(descritores_ia_descritores_fora_pulmao)[1], 2)
        #   self.padroesIndeterminados = criaPadroes(leDicionario('Descritores.xlsx', 'DESCRITORES QUALITATIVOS INDET')[1], 2)
          self.padroesIndeterminados = criaPadroes(leDicionario(descritores_ia_qualitativos_indet)[1], 2)
        #   self.padroesSecundarios = criaPadroes(leDicionario('Descritores.xlsx', 'DESCRITORES ESTRUTURAIS SEC')[1], 2)
          self.padroesSecundarios = criaPadroes(leDicionario(descritores_ia_estruturais_sec)[1], 2)
        #   self.padroesComparativos = criaPadroes(leDicionario('Descritores.xlsx', 'DESCRITORES COMPARATIVOS')[1], 2)
          self.padroesComparativos = criaPadroes(leDicionario(descritores_ia_descritores_comparativos)[1], 2)
          self.padroesComparativosClassif = criaPadroes({'COMPARACAO': ['estudo', 'laudo', 'exame', 'tc', 'ct', 'tomografia', 'resson.ncia', 'pet-ct', 'pet']}) 
          self.descritoresNodulares()
          self.leDescritoresQualitativos()
          self.leDescritoresConduta()

      def leFrasesPadrao(self):
        #   self.frasesPadrao = leDicionario('Descritores.xlsx', 'FrasesPadrao')[1]['FRASEPADRAO']
          self.frasesPadrao = leDicionario(descritores_ia_frasespadrao)[1]['FRASEPADRAO']
          for i in range(len(self.frasesPadrao)):
              self.frasesPadrao[i] = removeAcentos(self.frasesPadrao[i])
 
      def leDescritoresQualitativos(self):
        #   dicQualitativos = leDicionario('Descritores.xlsx', 'DESCRITORES QUALITATIVOS M OU S')[1]
          dicQualitativos = leDicionario(descritores_ia_descritores_qualitativo_m_ou_s)[1]
          self.suspeitosSimples = ['ONCOLOGICO', 'INDETERMINADO', 'INESPECIFICO', 'ESPICULADO', 'MICROCISTO', 'ENDOBRONQUICO', 'CAVITAÇÃO', 'MICROCISTOS', 'QNT. SUSPEITA', 'INTERMEDIARIO', 'SOLIDO', 'SUBSOLIDO', 'DESC. BENIGNA', 'BAIXA RELEV. CLINICA', 'CALCIFICADO', 'NAO CALCIFICADO', 'MORFOLOGIA IND', 'INFECCAO', 'PROCEDIMENTO']
          tempDict = {}
          for chave in self.suspeitosSimples:
              if chave in dicQualitativos:
                 tempDict[chave] = dicQualitativos[chave]

          chave = 'NAO ONCO'
          tempDict[chave] = dicQualitativos[chave]

          self.padroesQualitativos = criaPadroes(tempDict, 2)
          self.padroesQualitativos.extend(criaPadraoConjunto('ABAULAMENTO', dicQualitativos['EXTPLEURA1'], dicQualitativos['EXTPLEURA2']))
          
        #   self.padroesQualitativos.extend(criaPadroes(leDicionario('Descritores.xlsx', 'DESCRITORES QUALITATIVOS BENIGN')[1], 2))
          self.padroesQualitativos.extend(criaPadroes(leDicionario(descritores_ia_descritores_qualitativo_benign)[1], 2))

      def leDescritoresConduta(self):
          if 0:
            #  dicConduta = leDicionario('Descritores.xlsx', 'DESCRITORES QUALITATIVOS DE CON')[1]
             dicConduta = leDicionario(descritores_ia_descritores_qualitativo_de_con)[1]
             tempDict = {}
             tempDict['PROSSEGUIMENTO'] = dicConduta['PROSSEGUIMENTO']
             self.padroesConduta = criaPadroes(tempDict, 2)
             self.padroesConduta.extend(criaPadraoConjunto('PROSSEGUIMENTO', dicConduta['SUGESTAO'], dicConduta['SEGUIMENTO']))
             self.padroesConduta.extend(criaPadraoConjunto('CONTROLE', dicConduta['CONTROLE'], dicConduta['ACOMPANHAMENTO']))
          else:
            #  padroesConduta = leDicionario('Descritores.xlsx', 'DESCRITORES QUALITATIVOS DE CON')[1]
             padroesConduta = leDicionario(descritores_ia_descritores_qualitativo_de_con)[1]
             tempDict = {}
             for chave in ['PROSSEGUIMENTO', 'CONTROLE', 'NAO CONTROLE', 'CORRELACAO CLINICA']:
                 if chave in padroesConduta:
                    tempDict[chave] = padroesConduta[chave]

             self.padroesConduta = criaPadroes(tempDict, 2)
             self.padroesConduta.extend(criaPadraoConjunto('PROSSEGUIMENTO', padroesConduta['SUGESTAO1'], padroesConduta['SUGESTAO2']))

      def procuraDescritorVidroFosco(self, texto, descritoresAchados):
          texto = texto.lower()
          if 'vidro fosco' in texto:
             if not 'halo em vidro fosco' in texto and not 'vidro fosco periférico' in texto and not 'vidro fosco na periferia' in texto:
                if not 'SOLIDO' in descritoresAchados and not 'SUBSOLIDO' in descritoresAchados:
                   return ['VIDRO FOSCO']
          return []

      def _procuraDescritores(self, texto, dicionario):
          texto = texto.lower()
          descritores = [] 
          for chave in dicionario.keys():
              lista = dicionario[chave]
              for palavra in lista:
                  if palavra in texto:
                     descritores.append(chave.upper())
                     break
          return descritores

      def procuraDescritoresEstruturais(self, texto):
          # descritores secundarios
          descritoresTemporarios = self._procuraDescritores(texto, self.descritoresSecundarios)
          self.descritoresAchados.extend(descritoresTemporarios)

          # descritor vidro fosco
          self.procuraDescritorVidroFosco(texto)
          
      def procuraDescritoresLaudo(self, texto):
          # descritores comparativos
          descritoresTemporarios = self._procuraDescritores(texto, self.descritoresComparativos)
          self.descritoresAchados.extend(descritoresTemporarios)

      def analisaLocucaoVerbal(self, span):
          filhos = self.filhosSpan(span)
          indices = []
          for token in span:
              indices.append(token.i) 
          for token in filhos:
              if token.dep_ == 'obj':
                 indices.append(token.i) 
          return span.doc[np.min(indices):np.max(indices)+1]
           
      def labelEvolucao(self, span):
          label = ''  
          for token in span:
              if token.ent_type_ in ['SURGIMENTO', 'PROGRESSAO', 'ESTABILIDADE', 'REDUÇÃO', 'RESOLUÇÃO']:
                 label = token.ent_type_
                 break
          return label
 
      def procuraSpan(self, span):
          sortedKeys = sorted(self.dicionarioConcatenado)
          for key in sortedKeys:
              spanBase = self.dicionarioConcatenado[key]
              if key[0] <= span[0].i and key[1] >= span[-1].i:
                 return spanBase
          return None

      def descritoresEvolucao(self, doc):
          achados = []
          evolucoes = []
          classes = []
          for ent in doc.ents:
             if ent.label_ in ['NODULO', 'NODULOS']:
                achados.append(ent)
             if ent.label_ in ['SURGIMENTO', 'PROGRESSAO', 'ESTABILIDADE', 'REDUÇÃO', 'RESOLUÇÃO']:
                evolucoes.append(self.analisaLocucaoVerbal(ent)) 

          for achado, evolucao in itertools.product(achados, evolucoes):
              span1 = self.procuraSpan(achado)
              span2 = self.procuraSpan(evolucao)
              if span1 == span2 or self.spansConectados(span1, span2, True, semLimites = True): 
                 evolucaoLabel = self.labelEvolucao(evolucao) 
                 if not evolucaoLabel in classes:
                    classes.append(evolucaoLabel)
                  # ... código existente ... daqui p cima nesta funcao
          print(f"[DEBUG descritoresEvolucao] Classes encontradas: {classes}")
            
          # >>>>> FORÇA ADIÇÃO DE 'PROGRESSAO' SE A FRASE CONTÉM "MAIOR EM RELAÇÃO" <<<<<
          texto = doc.text.lower()
          if 'maior em relação' in texto or 'aumentou em relação' in texto:
              if 'PROGRESSAO' not in classes:
                  classes.append('PROGRESSAO')
                  print("[DEBUG] Forçando adição de 'PROGRESSAO' por palavras-chave")
     
          return classes

      def testaDescritor(self, sent, descritor):
          for ent in sent.ents:
              if ent.label_ == descritor:
                 candidate_labels = ['tem ' + ent.text, 'não tem ' + ent.text]
                 return self.verificaClasse(sent.text, candidate_labels, 'tem ' + ent.text, multi_label = False)
          return False 
 
      def classificaCalcificacao(self, fraseCalcificacao):
          texto = removeAcentos(fraseCalcificacao) 
          if 'calci' in texto:
             candidate_labels = ['calcificado', 'não calcificado', 'nao calcificado']
             return self.verificaClasse(fraseCalcificacao, candidate_labels, 'calcificado', multi_label = False)
          else:
             return True

      def caracterizaDocumento(self, doc, caracterizacoes):
                 for ent in doc.ents:
                     if ent.label_ in self.suspeitosSimples and not ent.label_ in caracterizacoes:
                        if ent.label_ == 'CALCIFICADO':
                           if self.classificaCalcificacao(doc.text):
                              caracterizacoes.append(ent.label_)
                           else:
                              caracterizacoes.append('NAO CALCIFICADO')
                        else:
                           caracterizacoes.append(ent.label_)

      def caracterizaNodulo(self, nodulo, doc):
                 for ent in doc.ents:
                     if ent.label_ in self.suspeitosSimples and not ent.label_ in nodulo.caracterizacoes:
                        if ent.label_ == 'CALCIFICADO':
                           if self.classificaCalcificacao(doc.text):
                              nodulo.caracterizacoes.append(ent.label_)
                           else:
                              nodulo.caracterizacoes.append('NAO CALCIFICADO')
                        else:
                           nodulo.caracterizacoes.append(ent.label_)

      def caracterizaNodulos(self):
          for nodulo in self.nodulos:
              # evolucao
              classes = self.descritoresEvolucao(nodulo.doc)
              if len(classes) > 0:
                 nodulo.caracterizacoes.extend(classes)
              # outros
              if 0:
                 self.caracterizaNodulo(nodulo, nodulo.doc)
              # vidro fosco
              nodulo.caracterizacoes.extend(self.procuraDescritorVidroFosco(nodulo.doc.text, nodulo.caracterizacoes))

      def xcaracterizaNodulos(self):
          for nodulo in self.nodulos:
              for key in self.dicionarioCaracterizacoes.keys():
                  frases = self.dicionarioCaracterizacoes[key]
                  probabilidade = 0
                  for frase in frases:
                      probabilidade = max(probabilidade, self.similaridadeFrases(nodulo.frase, frase))
                  if probabilidade >= thresholdSimiliridade:
                     nodulo.caracterizacoes.append(key)

      def classificaFraseComparacao(self, frase):
          classes = ['progressao', 'diminuição', 'resolução', 'estabilidade', 'surgimento']
          resultado = self.sentence_classification(frase, classes, multi_label=True)
          saidas = []
          for label, score in zip(resultado['labels'], resultado['scores']):
              if score > 0.9:
                 saidas.append(label) 

          # corrige bias a favor da diminuicao
          if 'diminuição' in saidas and len(saidas) > 1:
             saidas.remove("diminuição")
          return saidas

      def xxcaracterizaNodulos(self):
          for nodulo in self.nodulos:
              nodulo.caracterizacoes.extend(self.classificaFraseComparacao(nodulo.frase))

      def analisaPadraoComparativo(self, doc, profundo = True):
          result = False
          threshold = 0.55
          if True: #len([ent for ent in doc.ents if ent.label_ in ['COMPARACAO', 'COMPARATIVOS']]) > 0:
             if len([ent for ent in doc.ents if ent.label_ == 'DATE']) > 0:
                result = True #self.probabilidadeComparacao(doc.text) > threshold
                if not result and profundo:
                   self.processaTexto(doc.text) 
                   dicionario = self.reuneChunks(self.dicionario)  
                   sortedKeys = sorted(dicionario)
                   sortedKeys.sort()
                   for key in sortedKeys:
                       posTags = [ent.pos_ for ent in dicionario[key]]
                       ents = [ent.label_ for ent in dicionario[key].ents]
                       #if ('COMPARACAO' in ents or 'COMPARATIVOS' in ents) and 'DATE' in ents:
                       if ('NOUN' in posTags) and ('DATE' in ents):
                          result = self.probabilidadeComparacao(dicionario[key].text) > threshold
                          if result:
                             break 
          return result
      
    #   def analisaPadraoComparativo(self, doc, profundo=True):
    #     # Palavras-chave que indicam comparação
    #       palavras_chave = [
    #           'comparação', 'comparativo', 'comparar', 'correlação', 'análise comparativa',
    #           'em relação', 'relação ao', 'anterior', 'prévio', 'último exame', 'estudo anterior'
    #       ]
        
    #       texto = doc.text.lower()
    #       tem_palavra_chave = any(palavra in texto for palavra in palavras_chave)
    #       ##> DATE é um rotulo de identidade nomeada (NER) verifica se ha padrao de entidade DATE
    #       ##> combinado com palavra chave para verificar referencia temporal
    #       tem_data = len([ent for ent in doc.ents if ent.label_ == 'DATE']) > 0

    #       # Retorna True APENAS se a frase tiver UMA palavra-chave E UMA data
    #       result = tem_palavra_chave and tem_data

    #       # A lógica "profundo" abaixo é a fonte dos falsos positivos. Vamos desativá-la.
    #       # if not result and profundo:
    #       #     ... (todo o bloco complexo que usa similaridade)

    #       return result

      def reuneAchados(self):
          achados = ''
          for nodulo in self.nodulos:
              if nodulo.descricao.strip() != '':
                 achados = achados + ' ' + nodulo.descricao
              for info in nodulo.informacoes:
                  if achados == '':
                     achados = info 
                  else: 
                     achados = achados + ' ' + info 
          return achados

      def reuneMedidas(self):
          medidas = ''
          for nodulo in self.nodulos:
              for info in nodulo.infoCores:
                  if medidas == '':
                     medidas = info.tamanho 
                  else: 
                     if info.tamanho.strip() != '':
                        medidas = medidas + ' / ' + info.tamanho 
          return medidas

      def filtraComparacoes(self, docSents, profundo = True):
          sents = []
          for sent in docSents:
              if self.analisaPadraoComparativo(sent, profundo):
                 sents.append(sent)
          return sents   
           
      def eFrasePadrao(self, sent):
          processada = removeAcentos(sent)
          for frase in self.frasesPadrao:
              if frase in processada:
                 return True
          return False 
               
      def converteTextoEmLinhas(self, texto):
          docSents = sentencizer(texto)
          lista = []
          for sentTxt in docSents:
              if not self.eFrasePadrao(sentTxt):
                 lista.append(self.nlp(sentTxt))
          return lista

      def filtraFrases(self, docSents, menorValor = 0.4): ##corrigir
            sents = []
            for sent in docSents:
                temAchado = False
                temTamanho = False
                for ent in sent.ents:
                    if ent.label_ in ['NODULO', 'NODULOS']:
                        temAchado = True
        
                    if ent.label_ == 'TAMANHO':
                        temTamanhoAux, tam = self.pegaTamanho(ent, menorValor)
                        temTamanho = temTamanho or temTamanhoAux
                         
                if temAchado and temTamanho and len(sent.text) > 0:
                   sents.append(sent)
            return sents
        

    #   def filtraFrases(self, docSents, menorValor=0.6):
    #       sents = []
    #       for sent in docSents:
    #           labels = {ent.label_ for ent in sent.ents}
    #           temAchado = ('NODULO' in labels) or ('NODULOS' in labels)
    #           if not temAchado:
    #               # fallback textual se NER falhar por qualquer motivo
    #               if re.search(r'\bn[óo]dulo(?:s)?\b', sent.text, flags=re.IGNORECASE):
    #                   temAchado = True

    #           temTamanho = False
    #           for ent in sent.ents:
    #               if ent.label_ == 'TAMANHO':
    #                   ok, _ = self.pegaTamanho(ent, menorValor)
    #                   temTamanho = temTamanho or ok

    #           if temAchado and temTamanho and sent.text.strip():
    #               sents.append(sent)
    #       return sents

   

      def sentence_classification(self, texto, classes, multi_label = False):
          return self.sentence_classifier(texto, classes, multi_label = multi_label)

      def verificaClasse(self, texto, classes, classeAVerificar, multi_label = False):
          resultado = self.sentence_classifier(texto, classes, multi_label = multi_label)
          return resultado['labels'][0] == classeAVerificar and resultado['scores'][0] > 0.5
    
      def classificaFrase(self, texto, classes, classDefault):
          resultado = self.sentence_classification(texto, classes)
          saida = resultado['scores'][0] > 0.5 and resultado['labels'][0] == classDefault
          return saida, resultado

      def similaridadeFrases(self, fraseBase, frase):
          sentences = [
                fraseBase,
                frase
          ]
          sentence_embeddings = self.sentence_model.encode(sentences, show_progress_bar = False)
          resultado = cosine_similarity([sentence_embeddings[0]], [sentence_embeddings[1]])[0][0]
          del sentence_embeddings
          return resultado

      def probabilidadeComparacao(self, frase):
          frasesBases = ['A análise comparativa com a TC de tórax do dia 28/01/2022 não evidencia alterações significativas.', 'Correlação com exame anterior, feito em 01/10/2011.', "Análise comparativa com exame de 11/02/2022.", 'Comparação com tomografia computadorizada do tórax de 01/02/2012.', 'Último exame disponível realizado no dia 03/03/2022.', 'que não estava presente no exame anterior de 22/02/2018']
          prob = 0
          for fraseBase in frasesBases:
              prob = max(prob, self.similaridadeFrases(fraseBase, frase))
          return prob

      def identificaEntidade(self, span):
          if self.debug:
             print('Span:', span)
          if len(span) > 0:
             doc = span[0].doc 
             for ent in doc.ents:
                 if span[0].i <= ent.start and ent.end <= span[-1].i + 1:
                    if ent.label_ in ['NODULO', 'NODULOS', 'REGIAO', 'TAMANHO', 'DATE', 'SUSPEITOS']: 
                       if self.debug: 
                          print('Span:', span, 'Range:', (span[0].i, span[-1].i), 'Texto ent.:', ent.text) 
                       return ent.label_
                    if ent.label_ in self.descritoresDoPulmao or ent.label_ in self.descritoresForaPulmao:
                       return 'REGIAO'
          return None

      def verificaAchado(self, span):
          return self.identificaEntidade(span) in ['NODULO', 'NODULOS']

      def verificaLocalizacaoSpan(self, span):
          return self.identificaEntidade(span) == 'REGIAO'

      def concatenaSpan(self, lastConcat, span):
          if lastConcat == '':
             lastConcat = str(span)
          else:     
             lastConcat = lastConcat + ' '  + str(span)
          return lastConcat

      def pegaTipoTexto(self, lastConcat):
          doc = self.nlp(lastConcat)
          lastType = self.pegaTipo(doc)
          return lastType
 
      def guardaInformacao(self, lastConcatSpan, lastType):
          lastConcat = str(lastConcatSpan)
          if lastType == 'achado':
             if self.noduloAtual.descricao == '' or self.noduloAtual.descricao == lastConcat: 
                self.noduloAtual.descricao = lastConcat
             else:
                noduloAtual = self.noduloAtual
                if self.noduloAtual.inicializado():
                   self.nodulos.append(self.noduloAtual)

                self.noduloAtual = nodule(self)
                self.noduloAtual.descricao = lastConcat
                self.noduloAtual.frase = noduloAtual.frase
                self.noduloAtual.doc = noduloAtual.doc
                if noduloAtual.inicializado():
                   self.noduloAtual.anterior = noduloAtual
                self.nodulos.append(self.noduloAtual)
          else:    
             self.insereInformacao(lastConcat, lastType)
          self.caracterizaNodulo(self.noduloAtual, lastConcatSpan)

      def pegaTipo(self, span):
              entType = self.identificaEntidade(span)
              if entType  in ['NODULO', 'NODULOS']:
                   actualType = 'achado' 
              elif entType == 'TAMANHO':
                   actualType = 'tamanho' 
              elif entType == 'REGIAO':
                   actualType = 'local' 
              elif entType == 'DATE':
                   actualType = 'data' 
              elif entType == 'SUSPEITOS':
                   actualType = 'suspeito' 
              else:
                   actualType = '' 
              return actualType

      def temFilhos(self, token):
          lista = [subtoken for subtoken in token.children]
          return len(lista) > 0

      def filhosSpan(self, span, semLimites = False):
          filhos = []
          for token in span:
              temp = self.revisaFilhos(token, semLimites = semLimites)
              filhos.extend(temp)
          return filhos

      def ligacao(self, filhos, span, qualquerConexao=False):
          for token in span:
              if token in filhos and (not token.is_punct and token.dep_ not in ['acl:relcl', 'acl', 'advcl', 'appos', 'obl', 'obi'] or (token.head.pos_ == 'VERB') or qualquerConexao):
                 return True
          return False     
   
      def verboSeparado(self, span1, span2): 
          filhos = self.filhosSpan(span1)
          if len(span1) == 1 and span1[0].pos_ == 'VERB' and len([filho for filho in filhos if filho.dep_ in ['obl', 'obi', 'obj'] and filho in span2]) > 0:
             return True
          return False

      def spansConectados(self, span1, span2, qualquerConexao=False, semLimites = False):
          if span1 is None or span2 is None:
             return False
          filhos = self.filhosSpan(span1, semLimites = semLimites)
          # caso do verbo isolado
          #if self.verboSeparado(span1, span2):
          #   return True
          ab = self.ligacao(filhos, span2, qualquerConexao)
          ba = False
          if not ab:
             filhos = self.filhosSpan(span2, semLimites = semLimites)
             ba = self.ligacao(filhos, span1, qualquerConexao)
          return ab or ba

      def filhoDaRaiz(self, span):
           if span is None:
              return False
            
           return len([tok for tok in span if tok.head.dep_ == 'ROOT' and tok.head != tok]) > 0 

      def procuraSpanPai(self, span, dicionario):
          heads = list(set([token.head for token in span if token.head not in span]))
          if len(heads) > 0:
             for key in dicionario.keys():
                 if heads[0] in dicionario[key]:
                    return key, dicionario[key]
             return None, None

      def EFolha(self, span):
          filhos = self.filhosSpan(span)
          fora = [token for token in filhos if token not in span]
          return len(fora) == 0
   
      def passoConcatenacao(self, dicionario, ordem=1):
          sortedKeys = sorted(dicionario)
          sortedKeys.sort()
          if ordem == 2:
             sortedKeys = sortedKeys[::-1]
          lastSpan = None
          lastKey = None
          count = len(sortedKeys)
          for key in sortedKeys:
              span = dicionario[key]
              if self.EFolha(span):
                 if lastSpan is not None:
                    if self.spansConectados(lastSpan, span, True) and (len(span) + len(lastSpan)) < 15:
                    # if self.spansConectados(lastSpan, span, True, semLimites=True) and (len(span) + len(lastSpan)) < 25:
                        dicionario.pop(lastKey)
                        dicionario.pop(key)
                        lastKey = (min(lastKey[0], key[0]), max(lastKey[1], key[1]))
                        lastSpan = span[0].doc[lastKey[0]:lastKey[1]+1]
                        dicionario[lastKey] = lastSpan 
                        continue 
              lastSpan = span
              lastKey = key
            
      def reuneChunks(self, dicionario):
          passo = 1
          count = 0
          while count != len(sorted(dicionario)):
                count = len(sorted(dicionario))  
                if self.debug:
                   print('Passo', passo)
                   self.imprimeConceitos(dicionario)
                passo += 1

                self.passoConcatenacao(dicionario, 1) 
                self.passoConcatenacao(dicionario, 2) 
          return dicionario
      
      def _concatenaConceitos(self, dicionario, etapa=3):
            novodicionario = {}
            sortedKeys = sorted(dicionario)
            if 1:
                sortedKeys.sort()
                sortedKeys = sortedKeys[::-1]
                lastSpan = None
                lastKey = None
                lastType = ''

                dentroParentesis = False

                for key in sortedKeys:
                    span = dicionario[key]

                    # fecha bloco se encontrar ')'
                    if ')' in str(span):
                        if lastSpan is not None:
                            novodicionario[lastKey] = lastSpan
                        lastType = ''
                        lastSpan = None
                        lastKey = None
                        dentroParentesis = True

                    if dentroParentesis:
                        actualType = ''
                    else:
                        actualType = self.pegaTipo(span)

                    ligados = self.spansConectados(span, lastSpan, etapa in [1, 4])
                    podeConectar = ((lastType == actualType) or (actualType == '' or lastType == '')) and ligados

                    # mantém possibilidade de juntar dentro de parênteses
                    podeConectar = dentroParentesis or podeConectar

                    # só tenta conectar se já houver um lastKey
                    podeConectar = podeConectar and lastKey is not None

                    if lastKey is not None:
                        # mesma lógica de vírgulas do seu código original
                        virgulasAtuais = span.text.count(',') + lastSpan.text.count(',')
                        novasVirgulas = span[0].doc[min(lastKey[0], key[0]):max(lastKey[1], key[1])+1].text.count(',')
                        podeConectar = podeConectar and (virgulasAtuais == novasVirgulas)

                        # >>>>>>>>>>>>>>>>>>>>>>>>>> PATCH DELIMITADOR DE SEÇÃO <<<<<<<<<<<<<<<<<<<<<<<<<<
                        # bloqueia concatenação se houver delimitadores claros entre os spans
                        if podeConectar and lastSpan is not None:
                            a = lastKey[1]   # fim do span anterior
                            b = key[0]       # início do span atual
                            ini = min(a, b) + 1
                            fim = max(a, b)
                            if ini < fim:  # evita slice vazio/negativo
                                texto_entre = span[0].doc[ini:fim].text.strip()
                                if any(delim in texto_entre for delim in ['-', ':', '•', '—', '–']):
                                    podeConectar = False
                        # >>>>>>>>>>>>>>>>>>>>>>>>> FIM DO PATCH DELIMITADOR <<<<<<<<<<<<<<<<<<<<<<<<<<<<

                    if podeConectar:
                        if lastType == '':
                            lastType = actualType

                        lastKey = (min(lastKey[0], key[0]), max(lastKey[1], key[1]))
                        lastSpan = span[0].doc[lastKey[0]:lastKey[1]+1]

                        if lastType == '':
                            lastType = self.pegaTipo(lastSpan)
                    else:
                        if lastSpan is not None:
                            novodicionario[lastKey] = lastSpan

                        lastType = actualType
                        lastSpan = span
                        lastKey = key

                    # reabre fluxo após parênteses
                    if '(' in str(span):
                        novodicionario[lastKey] = lastSpan
                        lastType = ''
                        lastSpan = None
                        lastKey = None
                        dentroParentesis = False

                if lastSpan is not None:
                    novodicionario[lastKey] = lastSpan

            return novodicionario


    #   def _concatenaConceitos(self, dicionario, etapa=3):
    #       novodicionario = {}
    #       sortedKeys = sorted(dicionario)
    #       if 1:
    #           sortedKeys.sort()
    #           sortedKeys = sortedKeys[::-1]
    #           lastSpan = None
    #           lastKey = None
    #           lastType = ''

    #           dentroParentesis =  False
        
    #           for key in sortedKeys:
    #              span = dicionario[key]
    #              if ')' in str(span):
    #                 if lastSpan is not None:
    #                    novodicionario[lastKey] = lastSpan 
    #                 lastType = ''
    #                 lastSpan = None
    #                 lastKey = None
    #                 dentroParentesis =  True

    #              if dentroParentesis:
    #                 actualType = ''
    #              else:  
    #                 actualType = self.pegaTipo(span)
                  
    #              ligados = self.spansConectados(span, lastSpan, etapa in [1, 4])
    #              podeConectar = ((lastType == actualType) or (actualType == '' or lastType == '')) and ligados 

    #              #podeConectar = podeConectar and max(len(span), len(lastSpan)) < 10
    #              podeConectar = dentroParentesis or podeConectar
    #              #podeConectar = podeConectar and not ((self.filhoDaRaiz(span) and len(span) > 1) and self.filhoDaRaiz(lastSpan)) 
    #              podeConectar = podeConectar and lastKey is not None 
    #              if lastKey is not None:
    #                 virgulasAtuais = span.text.count(',') + lastSpan.text.count(',')
    #                 novasVirgulas = span[0].doc[min(lastKey[0], key[0]):max(lastKey[1], key[1])+1].text.count(',')
    #                 podeConectar = podeConectar and virgulasAtuais == novasVirgulas 
                    
    #              if podeConectar:
    #                 if lastType == '':
    #                    lastType = actualType
 
    #                 lastKey = (min(lastKey[0], key[0]), max(lastKey[1], key[1]))
    #                 lastSpan = span[0].doc[lastKey[0]:lastKey[1]+1]
 
    #                 if lastType == '':
    #                    lastType = self.pegaTipo(lastSpan)
    #              else:
    #                 if lastSpan is not None:
    #                    novodicionario[lastKey] = lastSpan 

    #                 lastType = actualType
    #                 lastSpan = span
    #                 lastKey = key

    #              if '(' in str(span):
    #                 novodicionario[lastKey] = lastSpan 
    #                 lastType = ''
    #                 lastSpan = None
    #                 lastKey = None
    #                 dentroParentesis =  False
                 
    #           if lastSpan is not None:
    #              novodicionario[lastKey] = lastSpan 

    #       return novodicionario
        
      def concatenaConceitos(self, dicionario, guardaInfo=True): 
          for etapa in [1,2,3, 4]:  
             tamanho = len(list(dicionario.keys()))+1
             while tamanho > len(list(dicionario.keys())):
                 tamanho = len(list(dicionario.keys()))
                 dicionario = self._concatenaConceitos(dicionario,etapa=etapa)
                 if etapa == 1:
                    break

          if guardaInfo:
             sortedKeys = sorted(dicionario)
             for key in sortedKeys:
                 span = dicionario[key]
                 actualType = self.pegaTipo(span)
                 #print('Span: ', span, 'Tipo: ', actualType)
                 self.guardaInformacao(span, actualType)

          return dicionario     

      def _apresentaResultado(self, nodulo):
          spacy.displacy.render(nodulo.doc, style='dep')     

          print('Frase  :', nodulo.frase)
          print('')
          print('Achado :', nodulo.descricao)
          nodulo.conteudo()
          print('Informacoes: ', nodulo.informacoes)
          print('Caracterizações: ', nodulo.caracterizacoes)
          print('')
          #print('Lista de conceitos')
          #self.imprimeConceitos(nodulo.dicionarioConcatenado)

          print('')
          print('')
          print('')
          

      def apresentaResultado(self):
          for nodulo in self.nodulos: 
              self._apresentaResultado(nodulo)
              
      def imprimeConceitos(self, dicionario):
          key = sorted(dicionario)
 
          for k in key:
             print(k, dicionario[k])     

      def geraSpan(self, token, interval): 
          minI = np.min(interval)
          maxI = np.max(interval)
          span = token.doc[minI:maxI+1]
          return minI, maxI, span

      def atualizaAnteriores(self, anterior, dicionario, dicionarioConcatenado):
          if anterior is not None:
             anterior.dicionario = dicionario
             anterior.dicionarioConcatenado = dicionarioConcatenado
             self.atualizaAnteriores(anterior.anterior, dicionario, dicionarioConcatenado)

      def criaListaConceitos(self, doc, guardaInfo=True):
          raiz, children = self.pegaRaiz(doc)
          self.dicionario = self.subtree(raiz)
          self.dicionario = self.verificaInconsistencias(self.dicionario)
          self.dicionarioConcatenado = self.concatenaConceitos(self.dicionario, guardaInfo=guardaInfo)
          if self.noduloAtual is not None:
             self.noduloAtual.dicionario = self.dicionario
             self.noduloAtual.dicionarioConcatenado = self.dicionarioConcatenado
             self.atualizaAnteriores(self.noduloAtual.anterior, self.dicionario, self.dicionarioConcatenado)  
    
      def verificaLocalizacaoNaoPulmao(self, localizacao):
          doc = self.nlp(localizacao)
          result = False
          for ent in doc.ents:
              if ent.label_ in self.descritoresForaPulmao:
                 result = True
                 break
          return result

      def verificaLocalizacaoPulmao(self, localizacao):
          doc = self.nlp(localizacao)
          result = False
          for ent in doc.ents:
              if ent.label_ in self.descritoresDoPulmao:
                 result = True
                 break
          return result

      def naoPulmao(self, nodulo):
           result = True
           for info in nodulo.infoCores:
               #result = result and (info.localizacao == '' or self.verificaLocalizacaoNaoPulmao(info.localizacao))
               result = result and self.verificaLocalizacaoNaoPulmao(info.localizacao)
               if not result:
                  break
           nodulo.naoPulmao = result

      def Pulmao(self, nodulo):
           result = True
           for info in nodulo.infoCores:
               if info.localizacao != '':
                 #result = result and (info.localizacao == '' or self.verificaLocalizacaoPulmao(info.localizacao))
                 result = result and self.verificaLocalizacaoPulmao(info.localizacao)
               if not result:
                  break
           nodulo.Pulmao = result
                      
      def printLog(self, *arg):
            msg = ''
            for a in arg:
                msg = msg + str(a) + ' '
            self.printOutput.append(msg)

    # original
    #   def pegaTamanho(self, ent, menorValor): 
    #         try:
    #            doc = ent.doc
    #            unidade = str(doc[ent.end-1])
    #            tam = 0
    #            for token in doc[ent.start:ent.end-1]:
    #                if token.like_num:
    #                   numero = str(token).replace(',', '.')
    #                   tam = max(tam, float(numero))

    #            valido = False
    #            if unidade == 'cm' or 'cent' in unidade:
    #                valido = tam >= menorValor
    #            elif unidade == 'mm' or 'mil' in unidade:
    #                tam = tam / 10
    #                valido = tam >= menorValor 
    #            else:
    #                print(ent)
    #                print(tam)
    #                print(unidade)
    #                for i in range(ent.start, ent.end+1):
    #                    print('Token ', i, doc[i])
    #                m
    #            temTamanho = valido
    #         except:
    #            temTamanho = False
    #            tam = 0 
    #         return temTamanho, tam
        
    #   def pegaTamanho(self, ent, menorValor): 
    #         try:
    #             doc = ent.doc
    #             unidade = str(doc[ent.end-1])
    #             numeros = []
                
    #             # Extrai todos os números na entidade
    #             for token in doc[ent.start:ent.end-1]:
    #                 if token.like_num:
    #                     numero = str(token).replace(',', '.')
    #                     numeros.append(float(numero))
                
    #             if len(numeros) == 0:
    #                 return False, 0
                
    #             # PEGA O PRIMEIRO NÚMERO (mais provável de ser a medida atual)
    #             tam = numeros[0]
                
    #             # Converte para cm se necessário
    #             valido = False
    #             if unidade == 'cm' or 'cent' in unidade:
    #                 valido = tam >= menorValor
    #             elif unidade == 'mm' or 'mil' in unidade:
    #                 tam = tam / 10
    #                 valido = tam >= menorValor 
    #             else:
    #                 # Assume cm se não especificado
    #                 valido = tam >= menorValor

    #             return valido, tam
    #         except Exception as e:
    #             print(f"[ERRO pegaTamanho] {e}")
    #             return False, 0

## testar
      def pegaTamanho(self, ent, menorValor): 
          try:
              # Pega o texto completo da entidade (ex: "0,8 cm")
              texto_tamanho = ent.text
              doc = ent.doc
              # Verifica unidade no token final ou no texto
              unidade = str(doc[ent.end-1]) if ent.end > ent.start else ""
              numeros = []
                
              # Extrai todos os números do texto
              import re
              matches = re.findall(r'[\d\.,]+', texto_tamanho)
              for match in matches:
                  try:
                      numero = float(match.replace(',', '.'))
                      numeros.append(numero)
                  except:
                      continue
              
              if len(numeros) == 0:
                  return False, 0
              
              # Maior medida
              tam = max(numeros)
               
              # Converte para cm se necessário
              valido = False
              if 'cm' in unidade.lower() or any(x in unidade.lower() for x in ['cent', 'centí', 'centi']):
                  valido = tam >= menorValor
              elif 'mm' in unidade.lower() or any(x in unidade.lower() for x in ['mil', 'milí', 'mili']):
                  tam = tam / 10  # Converte mm para cm
                  valido = tam >= menorValor
              else:
                  # Tenta inferir unidade pelo texto
                  if 'cm' in texto_tamanho.lower():
                      valido = tam >= menorValor
                  elif 'mm' in texto_tamanho.lower():
                      tam = tam / 10
                      valido = tam >= menorValor
                  else:
                      # Assume mm se não especificado (padrão em laudos)
                      tam = tam / 10
                      valido = tam >= menorValor

              return valido, tam
          except Exception as e:
              print(f"[ERRO pegaTamanho] {e}")
              return False, 0
      ## usada 06-10
    #   def pegaTamanho(self, ent, menorValor):
          
    #       try:
    #           # PEGA O TEXTO COMPLETO DA ENTIDADE
    #           texto_tamanho = ent.text
    #           # EXTRAI O PRIMEIRO NÚMERO + UNIDADE
    #           import re
    #           padrao = re.compile(r"([\d\.,]+)\s*(cm|mm)", flags=re.IGNORECASE)
    #           match = padrao.search(texto_tamanho)
    #           if match:
    #               numero_str = match.group(1).replace(',', '.')
    #               unidade = match.group(2)
    #               valor = float(numero_str)
    #               # CONVERTE PARA CM SE NECESSÁRIO
    #               if unidade.lower() == 'cm':
    #                   valor = valor * 10  # cm → mm
    #               # VALIDA PELO THRESHOLD
    #               valido = valor >= (menorValor * 10)  # threshold em mm
    #               return valido, valor
    #           else:
    #               return False, 0
    #       except Exception as e:
    #           print(f"[ERRO pegaTamanho] {e}")
    #           return False, 0
    
      def espacos(self, nivel):
            numEspacos = nivel * 3
            return numEspacos * ' '
        
      def verificaEntidade(self, span):
          for ent in self.ents:
              if span[0].i <= ent.start and ent.end <= span[-1].i + 1:
                 return ent.label_
          return None

      def verificaLocalizacao(self, texto):
          textoL = texto.lower()
          localizado = False
          for loc in self.locs:
               if str(loc) in textoL:
                  localizado = True
                  break
          return localizado

      def verificaTamanho(self, texto):
          eTamanho = False
          for tamanho in self.tamanhos:
               if str(tamanho) in texto:
                  eTamanho = True
                  break
          return eTamanho
     
      def insereInformacao(self, text, tipo = None):
          text = str(text)
          if text == 'None':
             return

          if tipo is None:
             if self.verificaTamanho(text):
                tipo = 'tamanho'
             if self.verificaLocalizacao(text):
                tipo == 'local'

          if tipo == 'achado':
             self.noduloAtual.descricao = text
          elif tipo == 'tamanho':
             self.noduloAtual.setaTamanho(text)
          elif tipo == 'local':
             self.noduloAtual.setaLocalizacao(text)
          else:  
             self.noduloAtual.setaInformacoes(text)

           
      def retornaFilhos(self, token, interval):
            if len(interval) == 0:
                return self.revisaFilhos(token)
            inicio = np.min(interval)
            fim = np.max(interval)+1
        
            children = []
            allchildren = self.revisaFilhos(token) 
            for subtoken in allchildren:
                if subtoken.i not in range(inicio, fim):
                   children.append(subtoken)
            return children
        
      def revisaFilhos(self, token, interval = None, semLimites = False):
            if interval is None or len(interval) == 0:
                inicio = -1
                fim = -2
            else:
                inicio = np.min(interval)
                fim = np.max(interval)+1

            #children = [subtoken for subtoken in token.children if subtoken.head == token and subtoken.i not in range(inicio, fim)]
            children = []
            doc = token.doc
            for tok in doc:
                if semLimites:
                   pertence = tok not in self.usados and tok in token.subtree and tok not in children and tok.i not in range(inicio, fim) and tok != token
                else:
                   pertence = tok not in self.usados and tok.head == token and tok not in children and tok.i not in range(inicio, fim) and tok != token

                if pertence:
                   children.append(tok)
            children.sort(key=lambda x: abs(token.i-x.i), reverse=False)
            return children

      def alteraLimites(self, limite_intervalo, i, limite_inferior):
          if limite_inferior:
             if limite_intervalo[0] == 0:
                limite_intervalo[0] = i
             else:
                limite_intervalo[0] = max(limite_intervalo[0], i) 
          else: 
             if limite_intervalo[1] == 0:
                limite_intervalo[1] = i
             else:
                limite_intervalo[1] = min(limite_intervalo[1], i) 
          return limite_intervalo      
            
      def pegaIntervalo_Old(self, token):
            interval = [token.i]
            children = self.revisaFilhos(token)
            tamanhoSubTree = len(list(token.subtree))
            for ent in token.doc.ents:
                if (token.i >= ent.start) and (token.i < ent.end):
                   interval.append(ent.start) 
                   interval.append(ent.end-1) 
                   for i in range(np.min(interval), np.max(interval)+1):
                      if token.doc[i] != token:
                         children.extend(self.retornaFilhos(token.doc[i], interval))
                
            oldChildren = children
            limite_intervalo = [0, 0]
            for subtoken in children:
                if subtoken.dep_ not in ['conj', 'obl:tmod', 'parataxis', 'nmod', 'appos', 'acl:relcl', 'acl', 'advcl', 'obj', 'obi', 'obl', 'nsubj'] and not subtoken.is_punct:
                   if self.temFilhos(subtoken):
                       subinterval, sublimite_intervalo = self.pegaIntervalo(subtoken) 
                       if np.sum(sublimite_intervalo) > 0:  
                          limite_intervalo = self.alteraLimites(limite_intervalo, subtoken.i, subtoken.i < token.i)
                       else:     
                          interval.extend(subinterval)
                   else:
                      if not subtoken.is_punct and abs(subtoken.i-token.i) < tamanhoSubTree:
                         interval.append(subtoken.i) 
                else:
                   limite_intervalo = self.alteraLimites(limite_intervalo, subtoken.i, subtoken.i < token.i)

            if np.sum(limite_intervalo) > 0:
                limites = limite_intervalo.copy() 
                if limites[0] == 0:
                   limites[0] = -1 

                if limites[1] == 0:
                   limites[1] = 10000000 

                interval = [i for i in interval if i < limites[1] and i > limites[0]]

            return interval, limite_intervalo

      def pegaIntervalo(self, token):
            interval = [token.i]
            children = self.revisaFilhos(token)
            tamanhoSubTree = len(list(token.subtree))
            # if token node is part of a entity preserve all entity text in one information chunk
            for ent in token.doc.ents:
                if (token.i >= ent.start) and (token.i < ent.end):
                   interval.append(ent.start) 
                   interval.append(ent.end-1) 
                   for i in range(np.min(interval), np.max(interval)+1):
                      if token.doc[i] != token:
                         children.extend(self.retornaFilhos(token.doc[i], interval))
                
            oldChildren = children
            limite_intervalo = [0, 0]
            for subtoken in children:
                   if self.temFilhos(subtoken) or subtoken.is_punct:
                      limite_intervalo = self.alteraLimites(limite_intervalo, subtoken.i, subtoken.i < token.i)
                   else:     
                      interval.append(subtoken.i)

            if np.sum(limite_intervalo) > 0:
                limites = limite_intervalo.copy() 
                if limites[0] == 0:
                   limites[0] = -1 

                if limites[1] == 0:
                   limites[1] = 10000000 

                interval = [i for i in interval if i < limites[1] and i > limites[0]]

            return interval, limite_intervalo

      def adicionaDicionario(self, dicionario, token, interval):
          key = None
          if len(interval) > 0:
             minI, maxI, span = self.geraSpan(token, interval)
             key = (minI, maxI, token)
             if len(interval) > 1 or span[0].text not in [',', '.', '/']:
                dicionario[key] = span
          return dicionario, key
    
      def subtree(self, token):
          dicionario = {}
          interval, limite = self.pegaIntervalo(token)
          dicionario, _ = self.adicionaDicionario(dicionario, token, interval)
          children = self.retornaFilhos(token, interval)
          for tok in children:
              subcon = self.subtree(tok) 
              for key in subcon.keys():
                  dicionario[key] = subcon[key] 
          return dicionario

      def overlapChaves(self, key1, key2):
          start1 = key1[0]; end1 = key1[1]
          start2 = key2[0]; end2 = key2[1]
          if start1 > start2:
              temp = start2
              start2 = start1
              start1 = temp

              temp = end2
              end2 = end1
              end1 = temp
            
          concatenado = (start1 <= start2 <= end1 or
             start1 <= end2 <= end1 or
             start2 <= start1 <= end2 or
             start2 <= end1 <= end2)

          return concatenado

      def verificaInconsistencias(self, dicionario):
          sortedKeys = sorted(dicionario)
          lastKey = None
          tok = None
          for key in sortedKeys:
              if tok is None:
                 tok = dicionario[key][0]
                
              if lastKey != None:
                 if self.overlapChaves(lastKey, key):
                    interval = [lastKey[0], lastKey[1], key[0], key[1]]
                    del dicionario[key]
                    del dicionario[lastKey]
                    dicionario, key = self.adicionaDicionario(dicionario, key[2], interval)
              lastKey = key 
          return dicionario 
        
      def pegaRaiz(self, doc):  
          raiz = None
          for token in doc:
              if token.dep_ == 'ROOT':
                 raiz = token
                 break

          if raiz is not None:
             children = self.revisaFilhos(raiz)
            
          return raiz, children
                    
      def inicializa(self):
          self.tamanhos = []
          self.locs = []
          self.usados = []
            
      def descobreEntidades(self, doc):
            for ent in doc.ents:
                if ent.label_ == 'TAMANHO':
                    self.tamanhos.append(ent)

                if ent.label_ == 'REGIAO':
                    self.locs.append(ent)

      def preprocessa(self, texto):
          texto = texto.replace('\n\xa0', '')
          texto = texto.replace('\n', ' ')

          texto = texto.replace('Nota:', '')
          texto = texto.replace('Notas:', '')
          texto = texto.replace('Análise:', '')

          texto = texto.replace('NOTA:', '')
          texto = texto.replace('NOTAS:', '')
          texto = texto.replace('ANÁLISES:', '')

          texto = texto.replace('"', '')
          texto = texto.strip()

          if texto[-1] == '.':
             texto = texto[:-1]
                
          self.inicializa()
          texto = texto.replace('\xA0', ' ').strip()
          self.printLog(texto)
          doc = self.nlp(texto)
          self.descobreEntidades(doc)
          self.ents = [ent for ent in doc.ents]
          return doc

      def preprocessaTurku(self, texto):
        ## tentativa de mudanca para spaCy
        #   docSpacy = self.preprocessa(texto) 
        #   # TUDO O QUE FAZEMOS AGORA É APLICAR O RULER DO SPA CY — NADA DE TURKU!
        #   doc = self.ruler(docSpacy)
        #   return doc
        # utilizacao do turku
          docSpacy = self.preprocessa(texto) 
          if self.tipoParser == 1:
             dataset = parseSpacyTokens(docSpacy, self.parser)
             doc = updateSpacyParser(docSpacy, dataset)
          elif self.tipoParser == 2:
             doc = self.nlp_udpipe(docSpacy)
          elif self.tipoParser == 3:
             spacyText = doc2text(docSpacy) 
             parsed = turkuParser.parse(spacyText)
             self.parsed = fixTurku(docSpacy, parsed)
             doc = turku2doc(self.parsed, self.nlp.vocab, 0)
          doc = self.ruler(doc)
          return doc

      def inicializaAchados(self):
          self.nodulos = []

      def processaTexto(self, texto):
          doc = self.preprocessaTurku(texto) 
          self.criaListaConceitos(doc, guardaInfo=False)
          return doc

      def processa(self, texto):
          self.noduloAtual = nodule(self)
          self.noduloAtual.frase = texto
          doc = self.preprocessaTurku(texto) 
          self.noduloAtual.doc = doc

          self.criaListaConceitos(doc)
          
          if self.noduloAtual.inicializado():
             self.nodulos.append(self.noduloAtual)

          pegaTamanho(self)
          i = 0
          while i < len(self.nodulos):
              nodulo = self.nodulos[i]
              if nodulo.tamanhoMaximo() == 0 and i > 0:
                 self.nodulos[i-1].concatenaInformacoes(nodulo)
                 self.nodulos.remove(nodulo)
              else: 
                 self.naoPulmao(nodulo)
                 self.Pulmao(nodulo) 
                 i = i + 1 

      def svg2PNG(self, doc, outFile, style='dep'):
          svgFile = os.path.splitext(outFile)[0] + '.svg' 
          svg = spacy.displacy.render(doc, style=style, jupyter=False)
          output_path = Path(svgFile)
          output_path.open("w", encoding="utf-8").write(svg)
          pngfile = os.path.splitext(outFile)[0] + '.png'
          os.system('inkscape -e %s %s' % (pngfile, svgFile))         
        
      def clearLog(self):  
           self.printOutput.clear()
          
      def saveLog(self, filename):  
           saveList(self.printOutput, filename)

      def finalizaTurku(self):
         turkuParser.send_final()
         turkuParser.join()      

In [0]:
class nodule:
      def __init__(self, ie):
          self.ie = ie
          self.descricao = ''
          self.informacoes = []
          self.infoCores = []
          self.caracterizacoes = []
          self.localizacaoNodulo = ''
          self.tamanhoNodulo = 0
          self.medidasNodulo = ''

          self.anterior = None
          self.dicionario = None
          self.dicionarioConcatenado = None

      def concatenaInformacoes(self, nodulo):
          if nodulo.descricao != '':
             self.descricao = self.descricao + ' / ' + nodulo.descricao
          self.informacoes.extend(nodulo.informacoes)
          self.infoCores.extend(nodulo.infoCores)
          self.caracterizacoes.extend(nodulo.caracterizacoes)
 
      def novoElementoCore(self): 
          self.infoCores.append(infoCore(self.ie)) 

      def setaTamanho(self, tamanho):
          if len(self.infoCores) == 0:
             self.novoElementoCore()

          if not self.infoCores[-1].setaTamanho(tamanho):
             self.novoElementoCore()
             self.infoCores[-1].setaTamanho(tamanho)

      def setaLocalizacao(self, localizacao):
          if len(self.infoCores) == 0:
             self.novoElementoCore()

          if not self.infoCores[-1].setaLocalizacao(localizacao):
             self.novoElementoCore()
             self.infoCores[-1].setaLocalizacao(localizacao)

      def conteudo(self):
          contador = 1
          for info in self.infoCores:
              print('Nodulo ', contador) 
              info.conteudo()
              contador += 1

      def setaInformacoes(self, texto):
          self.informacoes.append(texto)

      def localizacao(self):
          if len(self.infoCores) == 0:
             return self.localizacaoNodulo

          localizacao = ''
          for info in self.infoCores:
              localizacao = localizacao + ' ' + info.localizacao
          return localizacao
    # original
      def reuneMedidas(self):
          if len(self.infoCores) == 0:
             return self.medidasNodulo

          medidas = ''
          for info in self.infoCores:
              if medidas == '':
                 medidas = info.tamanho 
              else: 
                 if info.tamanho.strip() != '':
                    medidas = medidas + ' / ' + info.tamanho 
          return medidas

      def tamanhoMaximo(self):
          if len(self.infoCores) == 0:
             return self.tamanhoNodulo

          tamanho = 0
          for info in self.infoCores:
              if tamanho < info.tamanhoMaximo:
                 tamanho = info.tamanhoMaximo   
          return tamanho

      def inicializado(self):
          return len([info for info in self.infoCores if info.inicializado()]) > 0

In [0]:
class classificaLaudos:
      # def __init__(self, ingles = False, threshold = 0.6, corrigeTodos = False):    #  conrado - pra mudar o tamanho do nódulo 
      def __init__(self, ingles=False, threshold=0.6, corrigeTodos=False, df_tipo_pt=None, df_fora_pulmao=None, df_risco_pt=None):
          #self.benignos = loadList('benignos.txt')
          self.benignos  = [] # procurar este arquivo
         #  self.classificacoesDict, dictPadroes = leDicionario('classificacao-e-localizacao.xlsx', 'tipo-PT') ## comentada aqui
          self.classificacoesDict, dictPadroes = leDicionario(df_classificacao_localizacao_tipoPT)  ## adicionado
         #  self.localizacoesDict, dictLocalizacoes  = leDicionario('classificacao-e-localizacao.xlsx', 'local-PT') ## comentada no original
         #  self.localizacoesDict, dictLocalizacoes  = leDicionario('Descritores.xlsx', 'DESCRITORES FORA PULMAO') ## comentada aqui
          self.localizacoesDict, dictLocalizacoes = leDicionario(descritores_ia_descritores_fora_pulmao) ## adicionado
         #  _, riscosDict = leDicionario('classificacao-e-localizacao.xlsx', 'RISCO-PT') ## comentada aqui
          _, riscosDict = leDicionario(df_classificacao_localizacao_riscoPT) ## adicionado
          self.listariscos = list(riscosDict.keys())
          self.listariscos.insert(0, '>16mm')
          self.turku = 1
          self.threshold = threshold
          self.dataExame = ''

          padroes = criaPadroes(dictPadroes)
          padroesRisco = criaPadroes(riscosDict)
          padroesLocal = criaPadroes(dictLocalizacoes)

          self.nlpClassif = preparaNER(padroes)
          self.nlpRisco = preparaNER(padroesRisco)
          self.nlpLocal = preparaNER(padroesLocal)

          self.ie = InformationExtraction()
          print(60*'*')
          print(60*'*')
          print('Criando corretor')
          print(60*'*')
          #self.corretor = criaCorretor() << descomentar quando tiver a resposta do dev

          self.corrigeTodos = corrigeTodos
          self.pulmao = 'PULMÃO' 
          self.descritoresLaudo = {}
          self.idade = 0
          self.ultimoExame = 0
          self.exameMaisAntigo = 0
          self.criaIdentificadorMenor()
          self.openAiAdapter = None
          #self.openAiAdapter = openAiAdapter() << descomentar quando tiver a resposta do dev
          #self.openAiAdapter.setaClassificador(self) << descomentar quando tiver a resposta do dev

      # adicionar aqui criterios de menor que  ###  conrado
      def criaIdentificadorMenor(self):
          padrao = criaPadrao2('MENOR', ['menor%', 'menor% que', 'menos que','quase de','perto de'])
          self.nlpMenor = preparaNER(padrao)

      def consertaTamanho(self, tupla):
          doc = self.nlpMenor(tupla[5])    
          for ent in doc.ents:
              if ent.label_ == 'MENOR':
                 tupla[6] = tupla[6]-1
                 return

      def pegaClassificacao(self, achado):
          ents = identificaEntidades(self.nlpClassif, achado)
          return ents
    
      def abreLaudo(self, arquivo):
          laudo = open(arquivo, 'rt', encoding='UTF-8') 
          data = laudo.read()
          data = data.replace(u'\xa0', ' ')
          laudo.close()
          return data

      def pegaLocalizacao(self, localizacao):
          ents = identificaEntidades(self.nlpLocal, localizacao)
          if len(ents)> 0:
             return ents
          else:
             return None

      def converteOpenAiNodulo2Nodulo(self, openAiNodulo):
          nodulo = nodule(self.ie)
          nodulo.descricao = openAiNodulo['NODULO']
          nodulo.frase = openAiNodulo['FRASE']
          nodulo.caracterizacoes = self.retornaCaracterizacoes(openAiNodulo['NODULO'] + ' ' + openAiNodulo['CARACTERISTICAS'])
          nodulo.localizacaoNodulo = openAiNodulo['LOCALIZACAO']
          nodulo.tamanhoNodulo = self.retornaTamanho(openAiNodulo['TAMANHO'])
          nodulo.medidasNodulo = openAiNodulo['TAMANHO']
          nodulo.naoPulmao = self.ie.verificaLocalizacoNaoPulmao(nodulo.localizacaoNodulo)
          return nodulo
         

      def classificaNodulosOpenAi(self, respostaOpenAi):
          tuplas = []
          openAiNodulos = respostaOpenAi['LAUDO']['NODULOS']
          for openAiNodulo in openAiNodulos:
              nodulo = self.converteOpenAiNodulo2Nodulo(openAiNodulo)
              tupla = self.criaTuplaDoNodulo(nodulo, nodulo.frase, nodulo.frase)
              tuplas.append(tupla)
          return tuplas 

      def criaTuplaDoNodulo(self, nodulo, sent, naocorrigido):
         if 'RANDOMICA' in nodulo.caracterizacoes:
            nodulo.caracterizacoes.remove('RANDOMICA')

         if 'CALCIFICADO' in nodulo.caracterizacoes and ('NAO CALCIFICADO' in nodulo.caracterizacoes):
            nodulo.caracterizacoes.remove('CALCIFICADO')

         if 'BAIXA RELEV. CLINICA' in nodulo.caracterizacoes and ('INDETERMINADO' in nodulo.caracterizacoes or 'NAO CALCIFICADO' in nodulo.caracterizacoes):
            nodulo.caracterizacoes.remove('BAIXA RELEV. CLINICA')

         if 'ONCOLOGICO' in nodulo.caracterizacoes:
            nodulo.caracterizacoes.remove('ONCOLOGICO')

         local = nodulo.localizacao() 
         regiao = 'DESCONHECIDO'
         if nodulo.Pulmao:
            regiao = self.pulmao
         if nodulo.naoPulmao:
            regiao = self.pegaLocalizacao(local)
            if regiao is not None:
               regiao = juntaLista(regiao)
            else:
               regiao = ''
         else: 
            regiaoTemp = self.pegaLocalizacao(nodulo.frase)
            if regiaoTemp is not None:
               regiao = juntaLista(regiaoTemp)
               nodulo.naoPulmao = regiao != self.pulmao

      
         tamanho = nodulo.tamanhoMaximo()

         achado = nodulo.descricao
         infos =  juntaLista(nodulo.informacoes)

         if infos != '':
            achado = achado + ' ' + infos  
         medidas = nodulo.reuneMedidas()

         if ('massa' in achado or 'lesão' in achado or 'lesões' in achado) and 'ESPICULADO' in nodulo.caracterizacoes:
            if 'ONCOLOGICO' not in self.descritoresLaudo:
               self.descritoresLaudo['ONCOLOGICO'] = []
               if nodulo.frase not in self.descritoresLaudo['ONCOLOGICO']:
                  self.descritoresLaudo['ONCOLOGICO'].append(nodulo.frase)

         classifList = self.pegaClassificacao(sent)
         #print('Classificacao', classifList)
         calcificado = False
         ## trecho original
         # for termo in self.benignos:
         #    calcificado = termo in classifList
         #    if calcificado:
         #       break
         # >>>>> CORREÇÃO: Força calcificado se 'CALCIFICADO' estiver em classifList <<<<<
         calcificado = 'CALCIFICADO' in classifList
         print(f"[DEBUG] calcificado corrigido: {calcificado}")  # ← DEBUG

         ents = identificaEntidades(self.nlpRisco, sent)
         if tamanho >= 16:
            ents.append('>16mm')

         if 'INDETERMINADO' in nodulo.caracterizacoes:
            ents.append('INDETERMINADO')

         if 'SURGIMENTO' in nodulo.caracterizacoes:
            ents.append('SURGIMENTO')

         if 'PROGRESSAO' in nodulo.caracterizacoes:
            ents.append('PROGRESSAO')


         if nodulo.naoPulmao:
            tipoCaso = 'B' ##<< fora do pulmao
         elif calcificado:
            fraseCalcificacao = achado
            if self.ie.classificaCalcificacao(fraseCalcificacao):
               tipoCaso = 'AC' ##<< calcificado
            else: 
               tipoCaso = 'A' ##<< pulmonar nao calcificado
               calcificado = False
               if 'CALCIFICADO' in ents:
                  ents[ents.index('CALCIFICADO')] = 'NÃO CALCIFICADO'

               if 'CALCIFICADO' in classifList:
                  classifList[classifList.index('CALCIFICADO')] = 'NÃO CALCIFICADO'
         elif nodulo.Pulmao:
            tipoCaso = 'A' ##<< pulmonar nao calcificado
         else:
            tipoCaso = 'C' ##<< nao classificado

         classif = juntaLista(classifList) 
         classif = juntaLista(list(set(nodulo.caracterizacoes)))

         casos = len(ents)
         if casos == 0:
            tipoEstudo = '%s' % tipoCaso
         else: 
            if tipoCaso == 'AC':
               tipoCaso == 'A'
            tipoEstudo = '%s%.2d' % (tipoCaso, casos)

         print(f"\n=== DEBUG FINAL - ANTES DE MONTAR TUPLA ===")
         print(f"[DEBUG] nodulo.Pulmao: {nodulo.Pulmao}")
         print(f"[DEBUG] nodulo.naoPulmao: {nodulo.naoPulmao}")
         print(f"[DEBUG] calcificado: {calcificado}")
         print(f"[DEBUG] classifList: {classifList}")
         print(f"[DEBUG] nodulo.caracterizacoes: {nodulo.caracterizacoes}")
         print(f"[DEBUG] ents (riscos): {ents}")
         print(f"[DEBUG] tamanho: {tamanho}")
         print(f"[DEBUG] tipoCaso: {tipoCaso}")
         print(f"[DEBUG] casos (número de riscos): {len(ents)}")
         print(f"[DEBUG] tipoEstudo FINAL: {tipoEstudo}\n")

         tupla = [tipoEstudo, str(naocorrigido), str(sent), local, regiao, medidas, tamanho, achado, classif, ents, nodulo, ()]
         return tupla

      def classificaNodulos(self, docSents):
         print(f"\n[DEBUG] Total de frases recebidas: {len(docSents)}")
         for i, s in enumerate(docSents):
            print(f"  Frase {i+1}: {s.text}")

         sents = self.ie.filtraFrases(docSents, menorValor=self.threshold)
         print(f"\n[DEBUG] Frases após filtro (threshold={self.threshold}): {len(sents)}")
         for i, s in enumerate(sents):
            print(f"  Frase filtrada {i+1}: {s.text}")

         tuplas = [] 
         if len(sents) > 0:
            for sent in sents:
               tuplaAtual = None
               self.ie.clearLog()
               if self.corrigeTodos:
                  sent = corrige(self.corretor, str(sent.text)) 
               else:
                  sent = sent.text 
               self.ie.inicializaAchados()
               doc = self.ie.processa(sent)

               naocorrigido = str(sent) 
               self.ie.caracterizaNodulos() 
               for nodulo in self.ie.nodulos: 
                  tupla = self.criaTuplaDoNodulo(nodulo, sent, naocorrigido)
                  #print(tupla)
                  if tuplaAtual is None:
                     tuplaAtual  = tupla
                  else:  
                     if 'AC' not in tupla[0] and tupla[0][0] == 'A':
                        if tuplaAtual[0][0] != 'A' or 'AC' in tuplaAtual[0]: 
                           tuplaAtual  = tupla
                        elif tuplaAtual[0] < tupla[0]:
                           tuplaAtual = tupla

               if tuplaAtual is not None:
                  tuplas.append(tuplaAtual) 

         return tuplas

      def maisGrave(self, tuplas):
          tuplaAtual = None
          for tupla in tuplas:
              if tuplaAtual is None:
                 tuplaAtual  = tupla
              else:  
                 if 'AC' not in tupla[0] and tupla[0][0] == 'A':
                    if tuplaAtual[0][0] != 'A' or 'AC' in tuplaAtual[0]: 
                       tuplaAtual  = tupla
                    elif tuplaAtual[0] < tupla[0]:
                       tuplaAtual = tupla
          return tuplaAtual 

      def contagemDescritores(self, sent, descritor):
          contador = 0
          for ent in sent.ents:
              if ent.label_ == descritor:
                 contador += 1   
          return contador  
              
   
      def filtraDescritoresLaudo(self, docSents):
          for sent in docSents:
              for ent in sent.ents:
                  if ent.label_ in ['ONCOLOGICO', 'PROSSEGUIMENTO', 'CONTROLE', 'INFECCAO', 'EXAME BIOPSIA']:
                     incluiDescritor = True
                     if (ent.label_ in ['INFECCAO', 'ONCOLOGICO']):
                        incluiDescritor = self.ie.testaDescritor(sent, ent.label_)

                     if (ent.label_ in ['EXAME BIOPSIA']):
                        incluiDescritor = self.contagemDescritores(sent, ent.label_) > 2

                     if incluiDescritor:
                        if ent.label_ not in self.descritoresLaudo:
                           self.descritoresLaudo[ent.label_] = []
                        if sent.text not in self.descritoresLaudo[ent.label_]:
                           self.descritoresLaudo[ent.label_].append(sent.text)

      def realizaCategorizacao(self, tuplas):
             descritoresLaudo = [key for key in self.descritoresLaudo.keys() if len(self.descritoresLaudo[key]) > 0] 
             for tupla in tuplas:
                 self.consertaTamanho(tupla)
                 if True: 
                    if 'INDETERMINADO' in tupla[10].caracterizacoes and tupla[6] >= 6:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 5 nódulo + descritor INDETERMINADO + nódulo maior que 6mm')

                    if 'INESPECIFICO' in tupla[10].caracterizacoes and tupla[6] >= 6:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 5 nódulo + descritor INESPECIFICO + nódulo maior que 6mm')
                       
                    if 'COMPARACAO' not in descritoresLaudo and 'ONCOLOGICO' in descritoresLaudo:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 1.2  nódulo + descritor oncológico + ausência de COMPARATIVO')

                    #if tupla[6] >= 6 and 'ESPICULADO' in tupla[10].caracterizacoes and 'COMPARACAO' not in descritoresLaudo:
                    if tupla[6] >= 6 and 'ESPICULADO' in tupla[10].caracterizacoes:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 2: nódulo maior ou igual a 6mm + ESPICULADO + ausência de COMPARATIVO')

                    if tupla[6] >= 6 and 'ESPICULADO' in tupla[10].caracterizacoes and 'COMPARACAO' in descritoresLaudo and (self.ultimoExame >= 6):
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 2.1: nódulo maior ou igual a 6mm + ESPICULADO + COMP > 6 meses')

                    #if tupla[6] > 8 and 'COMPARACAO' in descritoresLaudo and (self.ultimoExame >= 6) and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                    if tupla[6] > 8 and (self.ultimoExame >= 6) and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 3: nódulo maior 8 mm + COMP >= 6 meses + ausência de CALCIFICAÇÃO ou BAIXA RELEV CLINICA')

                    #if tupla[6] > 6 and 'COMPARACAO' in descritoresLaudo and (self.ultimoExame >= 6) and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                    if tupla[6] > 6 and (self.ultimoExame >= 6) and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 3A: nódulo maior 6 mm + COMP >= 6 meses + ausência de CALCIFICAÇÃO ou BAIXA RELEV CLINICA')

                    #if tupla[6] > 8 and 'COMPARACAO' not in descritoresLaudo and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                    if tupla[6] > 8 and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 3.1: nódulo maior 8 mm + ausência de COMPARATIVO, CALCIFICAÇÃO ou BAIXA RELEV CLINICA')

### VERIFICAR
                    #if tupla[6] > 6 and 'COMPARACAO' not in descritoresLaudo and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                    if tupla[6] > 6 and 'BAIXA RELEV. CLINICA' not in tupla[10].caracterizacoes and 'CALCIFICADO' not in tupla[10].caracterizacoes:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 3.2: nódulo maior 6 mm + ausência de COMPARATIVO, CALCIFICAÇÃO ou BAIXA RELEV CLINICA')

                    if 'SURGIMENTO' in tupla[10].caracterizacoes and (self.ultimoExame >= 6):
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 4: nódulo + SURGIMENTO + COMP > 6 meses')

                    if 'PROSSEGUIMENTO' in descritoresLaudo:
                       tupla[11] = ('PROSSEGUIMENTO', 'Prosseguimento 1: descritor PROSSEGUIMENTO no laudo')

                 if len(tupla[11]) == 0:
                    if 'COMPARACAO' in descritoresLaudo:
                       if 'INDETERMINADO' in tupla[10].caracterizacoes and (('SURGIMENTO' in tupla[10].caracterizacoes or 'PROGRESSAO' in tupla[10].caracterizacoes) and 'ONCOLOGICO' in descritoresLaudo):
                          tupla[11] = ('CASO ACOMPANHADO', 'descritor de nódulo indeterminado + descritor de surgimento ou aumento + descritor oncológico')

                       elif (('SURGIMENTO' in tupla[10].caracterizacoes or 'PROGRESSAO' in tupla[10].caracterizacoes) and 'ONCOLOGICO' in descritoresLaudo):
                          tupla[11] = ('CASO ACOMPANHADO', 'descritor de surgimento ou aumento + descritor oncológico')
                      
                       elif ('ONCOLOGICO' in descritoresLaudo):
                          tupla[11] = ('CASO ACOMPANHADO', 'descritor oncológico')

                       elif (('ESTABILIDADE' in tupla[10].caracterizacoes or 'REDUCAO' in tupla[10].caracterizacoes) and (self.ultimoExame >= 6)):
                          tupla[11] = ('CASO ACOMPANHADO', 'descritor de estabilidade ou redução + COMP mais antigo > 6 meses')

                       elif ('ESTABILIDADE' in tupla[10].caracterizacoes or 'REDUCAO' in tupla[10].caracterizacoes):
                          tupla[11] = ('CASO ACOMPANHADO', 'descritor de estabilidade ou redução')

                    if tupla[6] < 6 and self.idade < 35:
                       tupla[11] = ('EXCLUSAO', 'Descritor de nódulo (<6 mm) + IDADE menor que 35 anos')

                    elif 'RESOLUCAO' in tupla[10].caracterizacoes:
                       tupla[11] = ('EXCLUSAO', 'Descritor de nódulo + descritor de resolução')

                    elif 'CALCIFICADO' in tupla[10].caracterizacoes and not 'ESPICULADO' in tupla[10].caracterizacoes and not 'INDETERMINADO' in tupla[10].caracterizacoes:
                       tupla[11] = ('EXCLUSAO', 'Descritor de nódulo + descritor de calcificação')

                    elif 'BAIXA RELEV. CLINICA' in tupla[10].caracterizacoes and not 'ESPICULADO' in tupla[10].caracterizacoes:
                       tupla[11] = ('EXCLUSAO', 'Descritor de nódulo + descritor de baixa relevância clínica')

                    #elif 'INDETERMINADO' not in tupla[10].caracterizacoes and 'ONCOLOGICO' in descritoresLaudo and 'INFECCAO' in descritoresLaudo:
                    #   tupla[11] = ('EXCLUSAO', 'Descritor de nódulo COM OU SEM descritor morfológico suspeito NA AUSÊNCIA DE descritor de nódulo indeterminado na frase E descritor oncológico no laudo e NA PRESENÇA DE descritor de infecção / inflamação no laudo')

                    elif self.ultimoExame >= 24 and ('ONCOLOGICO' in descritoresLaudo or 'SURGIMENTO' in tupla[10].caracterizacoes or 'PROGRESSAO' in tupla[10].caracterizacoes):
                       tupla[11] = ('EXCLUSAO', 'Paciente com exame COMP há mais de 24 meses contendo <descritor de nódulo> + E/OU <descritor oncológico> E/OU <descritor de surgimento ou aumento>')

                 if False: #len(tupla[11]) == 0:
                    if (self.idade >= 50) and ('ESPICULADO' in tupla[10].caracterizacoes) and (self.pegaLobo(tupla[10])[0] == 'SUPERIOR'):
                       tupla[11] = ('ALTO RISCO', '>=50 anos, Descritor morfológico suspeito, LOBO SUPERIOR')

                    elif ('ESPICULADO' in tupla[10].caracterizacoes) and (not 'COMPARACAO' in descritoresLaudo) and (not 'INFECCAO' in descritoresLaudo):
                       tupla[11] = ('ALTO RISCO', 'Nódulo espiculado com ausencia de COMPARACAO e INFECCAO')

                 if len(tupla[11]) == 0:
                    tupla[11] = ('EXCLUSAO', 'Nenhuma caracteristica relevante encontrada')

                 if 'EXAME BIOPSIA' in descritoresLaudo:
                    tupla[11] = ('EXCLUSAO', 'Caso BIOPSIA.')

                 if tupla[0][0] != 'A':
                    tupla[11] = ('EXTRAPULMONAR', 'FORA DO PULMAO')


      def textualizaDescritoresLaudo(self): 
          texto = ''
          for key in self.descritoresLaudo.keys():
              if len(self.descritoresLaudo[key]) > 0:
                 for frase in self.descritoresLaudo[key]:
                     if texto == '': 
                        texto = key + ' : ' + str(frase)
                     else:
                        texto = texto + '\n' + key + ' : ' + str(frase)

          return texto.replace('\\', '')

      def pegaLobo(self, nodulo):
          doc = self.ie.nlp(nodulo.frase)
          ents = [ent.label_ for ent in doc.ents]
          lobo = ''
          pulmao = ''
          if 'LOBO SUPERIOR' in ents:
              lobo = 'SUPERIOR'

          if 'LOBO MEDIO' in ents:
              lobo = 'MEDIO'

          if 'LOBO INFERIOR' in ents:
              lobo = 'INFERIOR'

          if 'PULMAO DIREITO' in ents:
              pulmao = 'DIREITO'

          if 'PULMAO ESQUERDO' in ents:
              pulmao = 'ESQUERDO'

          if 'SEGMENTO LATERAL' in ents or 'SEGMENTO MEDIAL'in ents:
              lobo = 'MEDIO'
              pulmao = 'DIREITO'

          if 'SEGMENTO SUPERIOR' in ents or 'SEGMENTO BASAL ANTEROMEDIAL' in ents or 'SEGMENTO BASAL MEDIAL' in ents or 'SEGMENTO BASAL LATERAL' in ents or 'SEGMENTO BASAL ANTERIOR'in ents or 'SEGMENTO BASAL POSTERIOR'in ents:
              lobo = 'INFERIOR'

          if 'SEGMENTO APICAL' in ents:
              lobo = 'SUPERIOR'

          if 'SEGMENTO APICOPOSTERIOR' in ents:
              lobo = 'SUPERIOR'
              pulmao = 'ESQUERDO'

          if 'SEGMENTO BASAL ANTEROMEDIAL' in ents:
              lobo = 'INFERIOR'
              pulmao = 'ESQUERDO'

          if 'LINGULA' in ents:
              pulmao = 'ESQUERDO'

          return lobo, pulmao 

      def pegaDescritoresLaudo(self, data, profundo = False):
          self.descritoresLaudo = {}
          docSents = self.ie.converteTextoEmLinhas(data)
          self.descritoresLaudo['COMPARACAO'] = self.ie.filtraComparacoes(docSents, profundo)
          self.filtraDescritoresLaudo(docSents)

          return self.descritoresLaudo

      def calculaDataUltimoExame(self):
          self.ultimoExame = 0
          
          # se nao for informadao a data do exame, ela sera considerada de hoje
          if self.dataExame == '' or self.dataExame is None:
             self.dataExame = datetime.today().strftime('%Y%m%d')

          if self.dataExame != '':
             if 'COMPARACAO' in self.descritoresLaudo: 
                for sent in self.descritoresLaudo['COMPARACAO']:
                    for ent in sent.ents:
                        if ent.label_ == 'DATE':
                           try:
                              anoTemp, mesTemp, diasTemp = ageCalculator2(converteData(string2Date(ent.text)), converteData(self.dataExame))
                              meses = abs(anoTemp * 12) + abs(mesTemp)
                              if diasTemp > 14:
                                 meses += 1
 
                              self.ultimoExame = meses
                           except Exception as e:
                              print('Data invalida', e)
                              
  
      def _classificaLaudo(self, data):
          self.descritoresLaudo = {} #<< restaurar
          docSents = self.ie.converteTextoEmLinhas(data) #<< restaurar
          tuplas = self.classificaNodulos(docSents) #<< restaurar

          print(">>> ENTROU EM _classificaLaudo <<<")  # 🔴 DEBUG FORÇADO << remover
         #  self.descritoresLaudo = {} # << remover
         #  docSents = self.ie.converteTextoEmLinhas(data) ## << remover
          print(f">>> Recebeu {len(docSents)} frases do converteTextoEmLinhas")  # 🔴 DEBUG << remover
         #  tuplas = self.classificaNodulos(docSents) ## << remover
          if 0:
             for tupla in tuplas:
                 print(tupla[10].caracterizacoes) 
          self.descritoresLaudo['COMPARACAO'] = self.ie.filtraComparacoes(docSents)
          self.filtraDescritoresLaudo(docSents)
          self.calculaDataUltimoExame()
          self.realizaCategorizacao(tuplas) 

          return tuplas, self.descritoresLaudo

      ##>> cai no _classificaLaudo pois openAiAdapter é False 
      def classificaLaudo(self, data, openAi=False):
          if openAi and self.openAiAdapter is not None:
             return self.openAiAdapter.processaLaudo(data)  
          else:
             return self._classificaLaudo(data)  

      def classificaLaudoOpenAi(self, data, openAiResponse):
          self.descritoresLaudo = {}
          docSents = self.ie.converteTextoEmLinhas(data)
          self.descritoresLaudo['COMPARACAO'] = self.ie.filtraComparacoes(docSents)
          self.filtraDescritoresLaudo(docSents)
          self.calculaDataUltimoExame()
          tuplas = self.classificaNodulosOpenAi(openAiResponse)
          self.realizaCategorizacao(tuplas) 

          return tuplas, self.descritoresLaudo

      def apresentaResultados(self):
          self.ie.apresentaResultado()

      def retornaTamanho(self, descricaoTamanho):
          doc = self.ie.nlp(descricaoTamanho)
          tam = 0
          for ent in doc.ents:
              if ent.label_ == 'TAMANHO':   
                 temTamanho, auxTam = self.ie.pegaTamanho(ent, 0)
                 if tam < auxTam:
                    tam = auxTam
          return tam * 10

      def retornaCaracterizacoes(self, descricao):
          doc = self.ie.nlp(descricao)
          caracterizacoes = []
          self.ie.caracterizaDocumento(doc, caracterizacoes)
          classes = self.ie.descritoresEvolucao(doc)
          if len(classes) > 0:
             caracterizacoes.extend(classes)
          caracterizacoes.extend(self.ie.procuraDescritorVidroFosco(doc.text, caracterizacoes))
          return caracterizacoes
          
      def processaFrase(self, frase):
          doc = self.ie.processa(frase)
          self.ie.caracterizaNodulos()
          self.apresentaResultados()
          return doc

In [0]:
classificador = classificaLaudos(threshold = 0.6, corrigeTodos = False)

####Dados de Entrada

In [0]:
print('catalogo:       ', CATALOG)
print('Tabela entrada: ', INPUT_TABLENAME)
print('Data Referencia:', data_execucao_modelo)

In [0]:
df_laudos_original_completo = spark.sql(f"""
                                select *
                                from {CATALOG}.{INPUT_TABLENAME}
                                where cast(dataExecucaoModelo as date) = date('{data_execucao_modelo}')
                                """).toPandas()
df_laudos_original_completo.shape

In [0]:
df_laudos_original_completo['data_limpa'] = df_laudos_original_completo['dataexame'].astype(str).str.replace('-', '') ##<<lake
df_laudos_original_completo['id'] = df_laudos_original_completo['data_limpa'] + df_laudos_original_completo['id_pct'] ##<< lake

df_laudos_original_completo = df_laudos_original_completo.dropna(subset=['data_limpa'])

if df_laudos_original_completo.shape[0] == 0:
    dbutils.notebook.exit("OK")
else:
    display(df_laudos_original_completo)

In [0]:
colunas = [
    'id',
    'id_pct',
    'cpf',
    'paciente',
    'idade_anos',
    'sexo',
    'nascimento',
    'idunidade',
    'unidade',
    'num_pedido_integracao',
    'nme_regional_hospital',
    'nme_convenio',
    'dataexame',
    'modalidade',
    'exame',
    'tipoexame',
    'an',
    'idstatus',
    #'exm_laudo_id',
    'DataLaudoLiberado',
    'Laudo',
    'dataExecucaoModelo',
    'telefonePacienteDDD',
    'telefonePaciente',
]

# reorganizar dataframe
df_laudos_original_completo = df_laudos_original_completo.reindex(columns=colunas)

df_laudos_original_completo.shape

In [0]:
df_laudos = df_laudos_original_completo.copy()

In [0]:
print(df_laudos.dataExecucaoModelo.min())
print(df_laudos.dataExecucaoModelo.max())

In [0]:
df_laudos['data_ref'] = pd.to_datetime(df_laudos['DataLaudoLiberado']).dt.date
df_laudos = df_laudos.rename(columns={'idade_anos':'idade_paciente', 'Laudo':'laudo_texto'})
df_laudos["an"] = df_laudos["an"].astype(str).str.zfill(16)
df_laudos = df_laudos.drop_duplicates(subset=["id", "dataexame", "laudo_texto", "an"], keep="first")

In [0]:
df_laudos.shape

In [0]:
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].apply(normaliza_nodulo)

In [0]:
df_laudos = df_laudos.fillna('vazio')

In [0]:
df_laudos['laudo_texto'] = df_laudos['laudo_texto'].apply(normaliza_laudo)

In [0]:
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('e no,', '')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace( ' cm - Fígado', ' cm. - Fígado')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('Pulmões: com atenuação preservada. Pequenos', 'Pulmões: com atenuação preservada, pequenos')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('algunsconfluentes', 'alguns confluentes')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('toracoabdominal', 'torax')

df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('a saber:', '')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('basesbases pulmonares', 'bases pulmonares')

df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('Pulmões:consolidações', 'Pulmões: consolidações')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('peribroncovascularesparsas', 'peribroncovascular esparsas')

df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('Notas:', '')

df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('nãocalcificados', 'não calcificados')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('nodulardiscretamente', 'nodular discretamente')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('Rarospequenos', 'Raros pequenos ')
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('aopacidade', 'a opacidade')
 
df_laudos["laudo_texto"] = df_laudos["laudo_texto"].str.replace('acometendopredominantemente ', 'acometendo predominantemente ')

####Classificador

In [0]:
resultados_nodulos = []    # Lista de nódulos (1 linha por nódulo)
resultados_laudos = []     # Lista de laudos (1 linha por laudo, com descritores globais)
    
for idx, row in df_laudos.iterrows():
    # texto = row['laudo_texto']
    texto_original = row['laudo_texto']
    texto_html_limpo = html2Text(texto_original)
    texto_limpo = retiraIlegais(texto_html_limpo)
    texto = normaliza_laudo(texto_limpo)
    idade = row.get('idade_paciente', 0) ## <- adicionado
    
    print(f"\n=== Processando laudo ID {row['id']} ===")
    print(f"Texto: {texto[:200]}...")

    classificador.idade = idade ## <- adicionado
    tuplas, descritores = classificador.classificaLaudo(texto)

    print(f"✅ Encontrou {len(tuplas)} nódulo(s)")
    print(f"📌 Descritores globais: {list(descritores.keys())}")

    descritores_texto = ""
    for chave, frases in descritores.items():
        for frase in frases:
            if descritores_texto:
                descritores_texto += "\n"
            descritores_texto += f"{chave} : {frase}"

    resultados_laudos.append({
        'id': row['id'],
        'exm_descritores_laudo': descritores_texto
    })

    for tupla in tuplas:
        frase_original = tupla[1] # frase do nódulo
        
        sub = frase_original
        info = extrai_medidas_generico(sub) ##<<add
        # info = extrai_medidas_generico_all(sub) 

        # Acesse os campos do dict:
        medidas_limpa   = info['texto']       if info else tupla[5]
        tamanho_mm_calc = info['tamanho_mm']  if info else tupla[6]

        resultados_nodulos.append({
            'id': row['id'],
            'id_pct': row['id_pct'],           
            'idunidade': row['idunidade'],
            'unidade': row['unidade'],
            'pct_nome': row['paciente'],
            'pct_cpf': row['cpf'],
            'idade_paciente': row['idade_paciente'],
            'pct_sexo': row['sexo'],
            'pct_datanasc': row['nascimento'],
            'telefonePacienteDDD': row['telefonePacienteDDD'],
            'telefonePaciente': row['telefonePaciente'],
            'exm_data': row['dataexame'],
            'exm_mod': row['modalidade'],
            'exm_titulo': row['exame'],
            'exm_tipo': row['tipoexame'],
            'num_pedido_integracao' : row['num_pedido_integracao'],
            'nme_regional_hospital' :row['nme_regional_hospital'],
            'nme_convenio' : row['nme_convenio'],
            'exm_an': row['an'],
            'exm_status': row['idstatus'],
            'exm_laudo_dataliber': row['DataLaudoLiberado'],
            'laudo_texto': row['laudo_texto'],
            'tipo_estudo': tupla[0],
            'frase_original': tupla[1],
            'frase_processada': tupla[2],
            'localizacao': tupla[3],
            'regiao': tupla[4],
            'medidas': tupla[5],
            'medidas_limpa': medidas_limpa,
            'tamanho_mm': tamanho_mm_calc,
            'descricao': tupla[7],
            'classificacoes': tupla[8],
            'riscos': tupla[9],
            'categorizacao': tupla[11],
            'data_ref': row['data_ref'],
            'dataExecucaoModelo': row['dataExecucaoModelo']
        })

In [0]:
len(resultados_nodulos)

In [0]:
# DataFrame de NÓDULOS (1 linha por nódulo)
df_nodulos = pd.DataFrame(resultados_nodulos)

# DataFrame de LAUDOS (1 linha por laudo, com descritores globais)
df_laudos_com_descritores = pd.DataFrame(resultados_laudos)

if not df_nodulos.empty:
    df_nodulos[['categoria', 'motivo']] = pd.DataFrame(
        df_nodulos['categorizacao'].tolist(), 
        index=df_nodulos.index
    )

In [0]:
df_laudos_com_descritores["exm_descritores_laudo"] = df_laudos_com_descritores.groupby("id")["exm_descritores_laudo"].transform(
    lambda x: [x.iloc[0]] + [None]*(len(x)-1)
)

In [0]:
df_nodulos["id"] = df_nodulos["id"].astype(str)
df_laudos_com_descritores["id"] = df_laudos_com_descritores["id"].astype(str)

df_laudos_unico = df_laudos_com_descritores.drop_duplicates(subset=["id"])

df_final = df_nodulos.merge(
    df_laudos_unico[["id", "exm_descritores_laudo"]],
    on="id",
    how="left"
)

# Preenche vazios
df_final['exm_descritores_laudo'] = df_final['exm_descritores_laudo'].fillna('')

In [0]:
df = agrupar_por_id_concatenando(df_final)

In [0]:
df = df.rename(columns={'id': 'record_id',
                        'laudo_texto':'exm_laudo_texto',
                        'tipo_estudo':'exm_class',
                        'frase_processada':'exm_frase_selec',
                        'localizacao':'exm_nodulo_local_detalhado',
                        'regiao':'exm_nodulo_local',
                        'medidas_limpa':'exm_nodulo_tamanho_detalhe',
                        'tamanho_mm':'exm_tamanho_mm',
                        'classificacoes':'exm_nodulo_tipo',
                        'descricao':'exm_nodulo_descricao',
                        'riscos':'exm_agravantes',
                        'categorizacao':'exm_justificativa_nlp',
                        })

In [0]:
col = (
    df['exm_justificativa_nlp']  
      .fillna('')                    
      .str.strip()                    
)

mask_exclusao = col.str.contains(
    r'^(EXCLUSAO|CASO ACOMPANHADO)\b',  
    case=False,                         
    regex=True
)

df['exm_encaminhamento_nlp'] = np.where(mask_exclusao, 'N', 'S')

In [0]:
df_limpo = limpa_descritores(df)

In [0]:
colunas = [
    'record_id', 
    'id_pct',
    'idunidade',
    'unidade',
    'pct_nome',
    'pct_cpf',
    'pct_sexo',
    'pct_datanasc',
    'idade_paciente',
    'telefonePacienteDDD',
    'telefonePaciente',
    'exm_data',
    'exm_mod',
    'exm_titulo',
    'num_pedido_integracao',
    'nme_regional_hospital',
    'nme_convenio',
    'exm_tipo',
    'exm_an',
    'exm_status',
    'exm_laudo_id',
    'exm_laudo_dataliber',
    'exm_laudo_texto',
    'exm_frase_selec',
    'exm_nodulo_local',
    'exm_nodulo_local_detalhado',
    'exm_tamanho_mm',
    'exm_nodulo_tamanho_detalhe',
    'exm_nodulo_tipo',
    'exm_nodulo_descricao',
    'exm_agravantes',
    'exm_class',
    'exm_descritores_laudo',
    'exm_justificativa_nlp',
    'exm_encaminhamento_nlp',
    'dataExecucaoModelo',
    'data_ref'
]

# reorganizar dataframe
df_limpo = df_limpo.reindex(columns=colunas)

In [0]:
ids_faltando = ids_nao_presentes(df_laudos, df_limpo)

print(len(ids_faltando))

In [0]:
df_limpo['exm_nodulo_local_limpo'] = df_limpo['exm_nodulo_local'].apply(remove_acentos).str.upper()

df_pulmao = df_limpo[df_limpo['exm_nodulo_local_limpo'].str.contains('PULMAO', na=False)]
df_sem_pulmao = df_limpo[~df_limpo['exm_nodulo_local_limpo'].str.contains('PULMAO', na=False)]

df_pulmao = df_pulmao.drop(columns='exm_nodulo_local_limpo')
df_sem_pulmao = df_sem_pulmao.drop(columns='exm_nodulo_local_limpo')

In [0]:
## laudos de entrada para classificacao

# display(df_laudos.sort_values('id_pct'))

In [0]:
## laudos classificados como pulmao

# display(df_pulmao.sort_values('id_pct'))

In [0]:
## laudos classificados para fora pulmao

# display(df_sem_pulmao.sort_values('id_pct'))

In [0]:
## apenas os laudos que nao receberam nenhuma classificacao

df_avalia_laudos = df_laudos[df_laudos['an'].isin(ids_faltando)]
# display(df_avalia_laudos.sort_values("id_pct"))

####Envio CSV

In [0]:
# # amostra com funcao
# df_pulmao_amostra = amostra(df_pulmao, total_amostras=500)
# df_sem_pulmao_amostra = amostra(df_sem_pulmao, total_amostras=500)
# df_avalia_laudos_amostra = amostra(df_avalia_laudos, total_amostras=500)

# # amostra com groupby
# df_pulmao_amostra = (
#     df_pulmao
#     .groupby(df_pulmao['exm_data'].astype('datetime64[ns]').dt.to_period('M'))
#     .apply(lambda x: x.sample(frac=0.5, random_state=42))
#     .reset_index(drop=True)
# )
# df_pulmao_amostra.shape

# df_pulmao_amostra = df_pulmao#.sample(n=500, random_state=42)
# df_sem_pulmao_amostra = df_sem_pulmao#.sample(n=500, random_state=42)
# df_avalia_laudos_amostra = df_avalia_laudos#.sample(n=500, random_state=42)

# print(df_pulmao_amostra.shape)
# print(df_sem_pulmao_amostra.shape)
# print(df_avalia_laudos_amostra.shape)

In [0]:
# colunas_limpeza = [
#     'exm_laudo_texto',
#     'exm_frase_selec',
#     'exm_nodulo_local_detalhado',
#     'exm_tamanho_mm',
#     'exm_nodulo_tamanho_detalhe',
#     'exm_nodulo_descricao',
#     'exm_descritores_laudo',
#     'exm_justificativa_nlp'
# ]

# colunas_limp = [
#     'exm_laudo_texto',
# ]

# df_pulmao_amostra = limpa_colunas_texto(df_pulmao_amostra, colunas_limpeza)
# df_sem_pulmao_amostra = limpa_colunas_texto(df_sem_pulmao_amostra, colunas_limpeza)
# df_avalia_laudos_amostra = limpa_colunas_texto(df_avalia_laudos_amostra, colunas_limp)

In [0]:
# df_pulmao_csv = envia_csv(df_pulmao_amostra, f'mod_pulmao_classificados_1_15_out_{data_execucao_modelo}')
# df_sem_pulmao_csv = envia_csv(df_sem_pulmao_amostra, f'mod_pulmao_fora_pulmao_classificados_1_15_out_{data_execucao_modelo}')
# df_avalia_laudos_csv = envia_csv(df_avalia_laudos_amostra, f'mod_pulmao_nao_classificados_1_15_out_{data_execucao_modelo}')

####Tabela saida

In [0]:
df_pulmao['nome_norm_lake'] = df_pulmao['unidade'].apply(normalizar_nome)

In [0]:
df_pulmao = df_pulmao.merge(
    depara_unidades[['id_unidade_lake', 'idUnidade_sf']],
    left_on='idunidade',
    right_on='id_unidade_lake',
    how='left'
)

df_pulmao = df_pulmao.drop(columns=['id_unidade_lake'])

In [0]:
if df_pulmao.shape[0] > 0:
    display(df_pulmao)

In [0]:
df_pulmao_match = df_pulmao[~df_pulmao['idUnidade_sf'].isna()].copy()
df_pulmao_no_match = df_pulmao[df_pulmao['idUnidade_sf'].isna()].copy()

In [0]:
if df_pulmao_no_match.shape[0] > 0:
    display(df_pulmao_no_match)

In [0]:
df_pulmao_no_match = df_pulmao_no_match.drop(columns=['idUnidade_sf'])

In [0]:
df_pulmao_no_match = df_pulmao_no_match.merge(
    depara_unidades[['idUnidade_sf', 'nome_norm_sf']],
    left_on='nome_norm_lake',
    right_on='nome_norm_sf',
    how='left'
)

In [0]:
if df_pulmao_no_match.shape[0] > 0:
    display(df_pulmao_no_match)

In [0]:
df_pulmao = pd.concat(
    [df_pulmao_match, df_pulmao_no_match],
    ignore_index=True
)

In [0]:
if df_pulmao.shape[0] > 0:
    display(df_pulmao)

In [0]:
faltantes_id = df_pulmao[df_pulmao['idUnidade_sf'].isna()]
if faltantes_id.shape[0] > 0:
    display(faltantes_id)

In [0]:
df_pulmao = df_pulmao.drop(columns=['nome_norm_lake', 'nome_norm_sf', 'data_ref'])

In [0]:
df_pulmao_envio = df_pulmao.copy()

In [0]:
# spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{OUTPUT_TABLENAME}""")
# spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{DESCRIPTO_TABLENAME}""")
# spark.sql(f"""DROP VIEW IF EXISTS {CATALOG}.vw_{OUTPUT_TABLENAME}""")
# spark.sql(f"""DROP VIEW IF EXISTS ia.vw_teste""")

In [0]:
# spark.sql(f"""
# ALTER TABLE ia.{OUTPUT_TABLENAME}
# ADD COLUMNS (
#   num_pedido_integracao STRING,
#   nme_regional_hospital STRING,
#   nme_convenio STRING)
# """)

# spark.sql(f"""
# ALTER TABLE {CATALOG}.{OUTPUT_TABLENAME}
# ADD COLUMN (idUnidade_sf STRING)
# """)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{OUTPUT_TABLENAME} (
  record_id STRING,
  id_pct STRING,
  idunidade STRING,
  idUnidade_sf STRING,
  unidade STRING,
  pct_nome STRING,
  pct_cpf STRING,
  pct_sexo STRING,
  pct_datanasc DATE,
  idade_paciente INT,
  telefonePacienteDDD STRING,
  telefonePaciente STRING,
  exm_data DATE,
  exm_mod STRING,
  exm_titulo STRING,
  exm_tipo STRING,
  num_pedido_integracao STRING,
  nme_regional_hospital STRING,
  nme_convenio STRING,
  exm_an STRING,
  exm_status STRING,
  exm_laudo_id STRING,
  exm_laudo_dataliber DATE,
  exm_laudo_texto STRING,
  exm_frase_selec STRING,
  exm_nodulo_local STRING,
  exm_nodulo_local_detalhado STRING,
  exm_tamanho_mm STRING,
  exm_nodulo_tamanho_detalhe STRING,
  exm_nodulo_tipo STRING,
  exm_nodulo_descricao STRING,
  exm_agravantes STRING,
  exm_class STRING,
  exm_descritores_laudo STRING,
  exm_justificativa_nlp STRING,
  exm_encaminhamento_nlp STRING,
  dataExecucaoModelo DATE
)
USING delta
COMMENT 'Laudos processados e enriquecidos com NLP para análise de nódulos pulmonares';
""")

In [0]:
df_class_pulmao_tb = spark.createDataFrame(df_pulmao_envio)

In [0]:
df_class_pulmao_tb.createOrReplaceTempView("vw_class_pulmao_tb")

In [0]:
spark.sql(f"""
INSERT INTO {CATALOG}.{OUTPUT_TABLENAME} (
    record_id,
    id_pct,
    idunidade,
    idUnidade_sf,
    unidade,
    pct_nome,
    pct_cpf,
    pct_sexo,
    pct_datanasc,
    idade_paciente,
    telefonePacienteDDD,
    telefonePaciente,
    exm_data,
    exm_mod,
    exm_titulo,
    exm_tipo,
    exm_an,
    exm_status,
    exm_laudo_id,
    exm_laudo_dataliber,
    exm_laudo_texto,
    exm_frase_selec,
    exm_nodulo_local,
    exm_nodulo_local_detalhado,
    exm_tamanho_mm,
    exm_nodulo_tamanho_detalhe,
    exm_nodulo_tipo,
    exm_nodulo_descricao,
    exm_agravantes,
    exm_class,
    exm_descritores_laudo,
    exm_justificativa_nlp,
    exm_encaminhamento_nlp,
    dataExecucaoModelo,
    num_pedido_integracao,
    nme_regional_hospital,
    nme_convenio
)
SELECT
    s.record_id,
    s.id_pct,
    s.idunidade,
    -- s.idUnidade_sf,
    regexp_replace(s.idUnidade_sf, '\\.0$', '') AS idUnidade_sf,
    s.unidade,
    s.pct_nome,
    s.pct_cpf,
    s.pct_sexo,
    to_date(s.pct_datanasc)                      AS pct_datanasc,         -- STRING -> DATE
    CAST(s.idade_paciente AS INT)                AS idade_paciente,       -- STRING -> INT
    s.telefonePacienteDDD,
    s.telefonePaciente,
    to_date(s.exm_data)                          AS exm_data,             -- STRING -> DATE
    s.exm_mod,
    s.exm_titulo,
    s.exm_tipo,
    s.exm_an,
    s.exm_status,
    CAST(s.exm_laudo_id AS STRING)               AS exm_laudo_id,         -- DOUBLE -> STRING
    to_date(s.exm_laudo_dataliber)               AS exm_laudo_dataliber,  -- STRING -> DATE
    s.exm_laudo_texto,
    s.exm_frase_selec,
    s.exm_nodulo_local,
    s.exm_nodulo_local_detalhado,
    s.exm_tamanho_mm,
    s.exm_nodulo_tamanho_detalhe,
    s.exm_nodulo_tipo,
    s.exm_nodulo_descricao,
    s.exm_agravantes,
    s.exm_class,
    s.exm_descritores_laudo,
    s.exm_justificativa_nlp,
    s.exm_encaminhamento_nlp,
    to_date(s.dataExecucaoModelo)                AS dataExecucaoModelo,   -- STRING -> DATE
    s.num_pedido_integracao,
    s.nme_regional_hospital,
    s.nme_convenio
FROM vw_class_pulmao_tb s
LEFT ANTI JOIN {CATALOG}.{OUTPUT_TABLENAME} t
    ON t.exm_an    = s.exm_an
""")

In [0]:
spark.sql(f"""
SELECT
  CAST(dataExecucaoModelo AS DATE) AS dataExecucao,
  COUNT(*) AS total_exames
FROM {CATALOG}.{OUTPUT_TABLENAME}
GROUP BY CAST(dataExecucaoModelo AS DATE)
ORDER BY dataExecucao DESC
""").display()

In [0]:
spark.sql(f"""
SELECT *
FROM {CATALOG}.{OUTPUT_TABLENAME}
WHERE cast(dataExecucaoModelo as date) = current_date()
""").display()

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{DESCRIPTO_TABLENAME} AS
SELECT
    *,
    CASE
      WHEN pct_nome IS NOT NULL
           AND pct_nome RLIKE '^[A-Za-z0-9+/=]+$'
           AND length(pct_nome) > 20          -- evita nomes curtos em texto puro
        THEN gold_corporativo.default.rdsl_decrypt(pct_nome, 1)
      ELSE pct_nome
    END AS pct_nom,

    CASE
      WHEN pct_cpf IS NOT NULL
           AND pct_cpf RLIKE '^[A-Za-z0-9+/=]+$'
           AND length(pct_cpf) > 20           -- idem, CPF criptografado normalmente é grandão
        THEN gold_corporativo.default.rdsl_decrypt(pct_cpf, 1)
      ELSE pct_cpf
    END AS pct_cp,

    CASE
      WHEN telefonePaciente IS NOT NULL
           AND telefonePaciente RLIKE '^[A-Za-z0-9+/=]+$'
           AND length(telefonePaciente) > 40  -- típico de Base64 longa
        THEN gold_corporativo.default.rdsl_decrypt(telefonePaciente, 0)
      ELSE telefonePaciente
    END AS telefonePacient

FROM {CATALOG}.{OUTPUT_TABLENAME}
WHERE nme_regional_hospital = 'SP' -- adicionado em 2026-01-12 para envios somente da reg SP
AND exm_encaminhamento_nlp = 'S' -- adicionado em 2026-02-21 a pedido
-- AND idunidade not in (147,200019,162,200022,69,100950,135,200146) -- desconsiderando ids novos e antigos (Vivalle - Mat Star - Vila Nova Star - Antonio Afonso)
-- em 19-03-26 foram adicionados para envio ao SF apenas as 21 unidades abaixo, o efeito sera nos envios de 20-03-26 em diante
AND idunidade in (873, -- Alphaville
                  200139,101, -- Bartira
                  200128,373, -- Campinas
                  200020,97, -- Central Leste
                  1353, -- Central Oeste
                  86, -- Central Sul
                  874, --Guarulhos
                  100968,67, -- ANALIA FRANCO
                  101043,42, -- SAO CAETANO
                  100970,103, -- ASSUNCAO - Sao Bernardo
                  200021,134, -- BRASIL MAUA
                  100999,5, -- SANTO ANDRE
                  16, --Tatuape
                  200023,257, -- SANTA ISABEL
                  200001,11, --  Itaim
                  200002,81, -- Morumbi
                  200130,54, -- Osasco
                  39, -- VILLA LOBOS
                  100965,100, -- Jabaquara
                  200129,62, --  RIBEIRAO PIRES
                  101027,141 -- IFOR
                  )
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{OUTPUT_TABLENAME_SF} AS
SELECT
    record_id,
    id_pct,
    idunidade,
    idUnidade_sf,
    unidade,
    pct_nom as pct_nome,
    pct_cp as pct_cpf,
    pct_sexo,
    pct_datanasc,
    idade_paciente,
    telefonePacienteDDD,
    telefonePacient as telefonePaciente,
    exm_data,
    exm_mod,
    exm_titulo,
    exm_tipo,
    nme_regional_hospital,
    nme_convenio,
    num_pedido_integracao,
    exm_an,
    exm_status,
    exm_laudo_id,
    exm_laudo_dataliber,
    exm_laudo_texto,
    exm_frase_selec,
    exm_nodulo_local,
    exm_nodulo_local_detalhado,
    exm_tamanho_mm,
    exm_nodulo_tamanho_detalhe,
    exm_nodulo_tipo,
    exm_nodulo_descricao,
    exm_agravantes,
    exm_class,
    exm_descritores_laudo,
    exm_justificativa_nlp,
    exm_encaminhamento_nlp,
    dataExecucaoModelo
FROM {CATALOG}.{DESCRIPTO_TABLENAME}
""")

In [0]:
spark.sql(f"""DROP TABLE IF EXISTS {CATALOG}.{DESCRIPTO_TABLENAME}""")

In [0]:
spark.sql(f"""
SELECT *
FROM {CATALOG}.{OUTPUT_TABLENAME_SF}
WHERE cast(dataExecucaoModelo as date) = current_date()
""").display()

In [0]:
spark.sql(f"""
SELECT
  CAST(dataExecucaoModelo AS DATE) AS dataExecucaoModelo,
  COUNT(*) AS total_exames
FROM {CATALOG}.{OUTPUT_TABLENAME_SF}
GROUP BY CAST(dataExecucaoModelo AS DATE)
ORDER BY dataExecucaoModelo DESC
""").display()

####Analise Resumida

In [0]:
def pct(num, den):
    return round((num / den) * 100, 2) if den else 0.0

# df_pulmao.loc[:, 'data_ref'] = pd.to_datetime(df_pulmao['dataExecucaoModelo']).dt.date
# df_sem_pulmao.loc[:, 'data_ref'] = pd.to_datetime(df_sem_pulmao['dataExecucaoModelo']).dt.date
# df_laudos.loc[:, 'data_ref'] = pd.to_datetime(df_laudos['dataExecucaoModelo']).dt.date
# df_avalia_laudos.loc[:, 'data_ref'] = pd.to_datetime(df_avalia_laudos['dataExecucaoModelo']).dt.date

df_pulmao['data_ref'] = pd.to_datetime(df_pulmao['dataExecucaoModelo']).dt.date
df_sem_pulmao['data_ref'] = pd.to_datetime(df_sem_pulmao['dataExecucaoModelo']).dt.date
df_laudos['data_ref'] = pd.to_datetime(df_laudos['dataExecucaoModelo']).dt.date
df_avalia_laudos['data_ref'] = pd.to_datetime(df_avalia_laudos['dataExecucaoModelo']).dt.date

datas = sorted(df_pulmao['data_ref'].unique())

linhas = []

for dia in datas:
    print("="*60)
    print(f"📅 Métricas do dia: {dia}")
    print("="*60)
    # Filtra cada DF pela data correspondente
    laudos_dia = df_laudos[df_laudos['data_ref'] == dia] ## total entrada
    pulm_dia   = df_pulmao[df_pulmao['data_ref'] == dia] ## laudos classificado no pulmao
    sem_dia    = df_sem_pulmao[df_sem_pulmao['data_ref'] == dia] ## laudos classificados fora pulmao
    aval_dia   = df_avalia_laudos[df_avalia_laudos['data_ref'] == dia] ## laudos que nao foram classificados

    # Calcula as métricas
    classificado            = pulm_dia['exm_an'].nunique() 
    classificado_sem_pulmao = sem_dia['exm_an'].nunique()
    total_nao_classificado  = aval_dia['an'].nunique()
    total_amostra           = laudos_dia['an'].nunique()
    soma_laudos             = classificado + classificado_sem_pulmao
    soma_laudos_total       = classificado + classificado_sem_pulmao + total_nao_classificado

    shape_classif              = pulm_dia.shape[0] + sem_dia.shape[0]
    shape_classif_pulmao       = pulm_dia.shape[0]
    shape_classif_fora_pulmao  = sem_dia.shape[0]
    shape_nao                  = aval_dia.shape[0] 
    shape_entrada              = laudos_dia.shape[0]
    
    encaminhado      = pulm_dia[pulm_dia['exm_encaminhamento_nlp'] == 'S']
    nao_encaminhado  = pulm_dia[pulm_dia['exm_encaminhamento_nlp'] == 'N']

    # percentuais (com proteção)
    perc_classificados        = pct(shape_classif, shape_entrada)
    perc_class_pulmao         = pct(shape_classif_pulmao, shape_entrada)
    perc_class_pulmao_sim     = pct(encaminhado.shape[0], shape_classif_pulmao)
    perc_class_pulmao_nao     = pct(nao_encaminhado.shape[0], shape_classif_pulmao)
    
    # Impressões
    # print("Id's únicos")
    # print("Total classificado:    ", soma_laudos)
    # print("Total não classificado:", total_nao_classificado)
    # print("Total da amostra:      ", total_amostra)
    # print("Soma dos laudos:       ", soma_laudos_total)
    
    print("\nShape dos Envios")
    print("Shape entrada:              ", shape_entrada)
    print("Shape não classificados:    ", shape_nao)
    print("Shape class total:          ", shape_classif)
    print("Shape class pulmao:         ", shape_classif_pulmao)
    print("Shape class fora pulmao:    ", shape_classif_fora_pulmao)
    print("Conferindo total:           ", shape_classif + shape_nao)
    
    print("\nPercentuais")
    print("Percentual total:       ", perc_classificados)
    print("Percentual Pulmao:      ", perc_class_pulmao)
    print("Percentual S:           ", perc_class_pulmao_sim)
    print("Percentual N:           ", perc_class_pulmao_nao)
    print("\n")

    # perc_classificados = round(shape_classif / laudos_dia.shape[0] * 100, 2)
    # perc_class_pulmao = round(shape_classif_pulmao / laudos_dia.shape[0] * 100, 2)
    # perc_class_pulmao_sim = round(encaminhado.shape[0] / shape_classif_pulmao * 100, 2)
    # perc_class_pulmao_nao = round(nao_encaminhado.shape[0] / shape_classif_pulmao * 100, 2)

    linhas.append({
        "dia": dia,
        "volumeEntrada": shape_entrada,
        "classificados": shape_classif,
        # "naoClassificados": shape_nao,
        "classificadosPulmao": shape_classif_pulmao,
        "classificadosForaPulmao": shape_classif_fora_pulmao,
        "percClass": perc_classificados,
        "percClassPulmao": perc_class_pulmao,
        "percClassPulmaoEncNLPSim": perc_class_pulmao_sim,
        "percClassPulmaoEncNLPNao": perc_class_pulmao_nao,     
    })

    # linhas.append({
    #     "dia": dia,
    #     "classificadosGeral": shape_classif,
    #     "classificadosPulmao": shape_classif_pulmao,
    #     "classificadosForaPulmao": shape_classif_fora_pulmao,
    #     "naoClassificados": shape_nao,
    #     "volumeEntrada": shape_entrada
    # })

In [0]:
df_metricas_pd = pd.DataFrame(linhas)

In [0]:
df_metricas_spark = spark.createDataFrame(df_metricas_pd)
df_metricas_spark = df_metricas_spark.withColumn("dia", F.to_date("dia"))
df_metricas_spark.createOrReplaceTempView("vw_metricas_spark")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{METRICS_TABLENAME} (
  dia DATE,
  volumeEntrada INT,
  classificados INT,
  classificadosPulmao INT,
  classificadosForaPulmao INT,
  percClass DOUBLE,
  percClassPulmao DOUBLE,
  percClassPulmaoEncNLPSim DOUBLE,
  percClassPulmaoEncNLPNao DOUBLE
)
USING delta
COMMENT 'Métricas diárias de classificação de laudos pulmão'
""")

In [0]:
spark.sql(f"""
INSERT INTO {CATALOG}.{METRICS_TABLENAME}
SELECT s.*
FROM vw_metricas_spark s
LEFT ANTI JOIN {CATALOG}.{METRICS_TABLENAME} t
  ON CAST(t.dia AS DATE) = CAST(s.dia AS DATE)
""")

In [0]:
spark.sql(f"""SELECT * FROM {CATALOG}.{METRICS_TABLENAME} ORDER BY dia DESC""").display()

In [0]:
dbutils.notebook.exit("OK")